# Optimizing Network-friendly Recommendations and Caching jointly, using Reinforcement Learning (Single Edge)

# Imports

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity


import random
from itertools import combinations,chain
import time
import copy
import math
from  scipy import sparse


from tqdm.notebook import trange, tqdm
# from tqdm import tqdm,trange


import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import os
import collections

import matplotlib.pyplot as plt
from functools import lru_cache

##### testing
from torch.backends import cudnn

cudnn.benchmark=False
# torch.use_deterministic_algorithms(True)

torch.__version__

# Collections

In [ ]:
Transition = collections.namedtuple('Transition',
                                    field_names=['state','next_state'])

Experience = collections.namedtuple('Experience',
                                    field_names=['state','action','reward',
                                                 'done','next_state','state_rep','next_state_rep','state_topN','state_topN_map','next_topN','next_topN_map'])

Result = collections.namedtuple('Result',
                                    field_names=['reward','cost',
                                                 'max_iter','time'])

# Utility Functions

## Numpy Utility

In [ ]:
def normalize_tensor(input_tensor):
    max=torch.max(input_tensor)
    min=torch.min(input_tensor)
    if min==max:
        return torch.ones_like(input_tensor)
    return (input_tensor-min)/(max-min)

def normalized_array(input_array):
    max=np.max(input_array)
    min=np.min(input_array)
    if min==max:
        return np.ones_like(input_array)
    return (input_array-min)/(max-min)



def normalize_ratings_schema(df, watch_ratio_clip=2.0):
    """Normalize a raw ratings dataframe to the canonical (userId, movieId, rating[,
    timestamp]) schema used by get_top_movies/create_u/create_popularity, auto-detecting
    the source format so any of those three functions can be pointed at either raw file
    directly (no separate per-dataset conversion step needed):
      - MovieLens: already (userId, movieId, rating) -- returned unchanged.
      - KuaiRec:   (user_id, video_id, watch_ratio) -- there is no explicit star rating,
                   so watch_ratio (play_duration/video_duration) is clipped at
                   `watch_ratio_clip` (it's heavy-tailed, p99~3.6) and used as the rating
                   analog. create_u mean-centers per item and cosine similarity is
                   invariant to any uniform affine rescale, so the pipeline is unaffected
                   by this choice beyond bounding outlier influence.
    Idempotent: normalizing an already-canonical frame is a no-op. Raises ValueError for
    an unrecognized schema (e.g. a synthetic/other file) so the caller gets a clear error
    instead of a KeyError deep inside the CF pipeline.
    """
    cols = set(df.columns)
    if {'userId', 'movieId', 'rating'}.issubset(cols):
        return df
    if {'user_id', 'video_id', 'watch_ratio'}.issubset(cols):
        out = df.rename(columns={'user_id': 'userId', 'video_id': 'movieId'}).copy()
        out['rating'] = out['watch_ratio'].clip(upper=watch_ratio_clip).astype(np.float32)
        keep = ['userId', 'movieId', 'rating'] + (['timestamp'] if 'timestamp' in out.columns else [])
        return out[keep]
    raise ValueError(
        f"Unrecognized ratings schema (columns={sorted(cols)}); expected MovieLens "
        f"(userId, movieId, rating) or KuaiRec (user_id, video_id, watch_ratio)."
    )


def get_top_movies(number_of_contents, dataset=None, df=None):
    if df is None:
        df = pd.read_csv(dataset)
    df = normalize_ratings_schema(df)
    if 'timestamp' in df.columns:
        df = df.drop('timestamp', axis='columns')
    top_movies = (
        df.groupby('movieId')['userId']
          .count()
          .sort_values(ascending=False)
          .head(number_of_contents)
          .index
          .tolist()          # fixed ordered list, e.g. [318, 296, 356, ...]
    )
    return top_movies        # index 0 → most-rated movie, index 1 → second, etc.



def create_u(number_of_contents, dataset=None, top_movies=None, df=None,
             k=10, min_common=2, user_chunk=2000, verbose=True):
    """Continuous content-similarity matrix U, float32 with values in [0, 1].

    No dataset  -> random U in [0, 1] (unchanged legacy behaviour).
    With dataset -> MovieLens collaborative-filtering recipe, STREAMED over users so
    only N x N content matrices ever stay resident (fits in tight RAM even on the
    full MovieLens-25M file):
      1. item-to-item CF (k=10 nearest items) predicts the missing user ratings;
      2. cosine similarity of each content pair on item-mean-centered ratings
         (each rating minus that item's average rating);
      3. return the similarity itself, clipped to [0, 1] (NO binary saturation).
    Returns an N x N float32 matrix (~100 MB at N=5000). `user_chunk` only trades
    speed vs. the transient per-chunk buffer; it never changes the N x N output.
    """
    N = number_of_contents
    if dataset is None and df is None:
        u = np.random.uniform(0, 1, (N, N)).astype(np.float32)
        np.fill_diagonal(u, 0)
        return np.array(u)

    if df is None:
        # Schema-aware compact read: try MovieLens columns, then KuaiRec columns, else
        # fall back to a full read (normalize_ratings_schema below raises a clear error
        # for anything else, instead of a KeyError deep inside the CF pipeline).
        try:
            df = pd.read_csv(dataset, usecols=['userId', 'movieId', 'rating'],
                             dtype={'userId': 'int32', 'movieId': 'int32', 'rating': 'float32'})
        except ValueError:
            try:
                df = pd.read_csv(dataset, usecols=['user_id', 'video_id', 'watch_ratio'],
                                 dtype={'user_id': 'int32', 'video_id': 'int32', 'watch_ratio': 'float32'})
            except ValueError:
                df = pd.read_csv(dataset)
    df = normalize_ratings_schema(df)
    if 'timestamp' in df.columns:
        df = df.drop('timestamp', axis='columns')
    if top_movies is None:
        top_movies = get_top_movies(number_of_contents, df=df)

    # --- Recode the ratings of the N selected contents into a sparse users x N matrix.
    df_f = df[df['movieId'].isin(top_movies)]
    movie_to_idx = {m: i for i, m in enumerate(top_movies)}
    m_idx = df_f['movieId'].map(movie_to_idx).to_numpy(dtype=np.int32)
    u_idx = df_f['userId'].astype('category').cat.codes.to_numpy(dtype=np.int32)
    rate  = df_f['rating'].to_numpy(dtype=np.float32)
    n_users = int(u_idx.max()) + 1
    R = sparse.csr_matrix((rate, (u_idx, m_idx)), shape=(n_users, N), dtype=np.float32)
    del df, df_f, m_idx, u_idx, rate            # free the ratings frame before the heavy phase

    counts = np.asarray((R != 0).sum(axis=0)).ravel().astype(np.float64)
    sums   = np.asarray(R.sum(axis=0)).ravel().astype(np.float64)
    item_mean   = np.where(counts > 0, sums / np.maximum(counts, 1.0), 0.0).astype(np.float32)
    global_mean = np.float32(sums.sum() / max(counts.sum(), 1.0))

    def _chunks():
        for s in range(0, n_users, user_chunk):
            yield s, min(s + user_chunk, n_users)

    # --- Pass 1: item-item similarity over the COMMON raters (item-mean-centered
    #     adjusted cosine), used only to pick each item's k nearest neighbours.
    #     B == A.T and the matrices are symmetric, so we accumulate just num, A, common.
    num    = np.zeros((N, N), dtype=np.float32)
    A      = np.zeros((N, N), dtype=np.float32)
    common = np.zeros((N, N), dtype=np.float32)
    for s, e in _chunks():
        Rc = R[s:e].toarray()
        M  = (Rc != 0).astype(np.float32)
        C  = (Rc - item_mean[None, :]) * M               # centered, 0 where missing
        num    += C.T @ C                                 # sum over common raters of c_i*c_j
        A      += (C * C).T @ M                           # A[i,j] = sum_u c_i^2 * 1[rated_j]
        common += M.T @ M
    with np.errstate(invalid='ignore', divide='ignore'):
        np.sqrt(A, out=A)                                 # A <- sqrt(A), so A.T == sqrt(B)
        denom = A * A.T
        np.divide(num, denom, out=num)                    # similarity stored in `num`
    S = num
    S[~np.isfinite(S)] = 0.0
    S[common < min_common] = 0.0
    np.fill_diagonal(S, 0.0)
    del A, common, denom

    # --- Keep only the k nearest POSITIVE neighbours per item.
    S[S < 0] = 0.0
    if k is not None and k < N:
        kth = -np.partition(-S, k - 1, axis=1)[:, k - 1]  # k-th largest similarity per row
        S[S < kth[:, None]] = 0.0
    St = np.ascontiguousarray(S.T)                        # St[:, i] = neighbours of item i
    del S, num

    # --- Pass 2: CF-complete the missing ratings and accumulate the Gram matrix of the
    #     completed ratings; the item-mean centering is folded in via the identity
    #     G_centered = G - colsum (x) colsum / n_users.
    G = np.zeros((N, N), dtype=np.float64)
    colsum = np.zeros(N, dtype=np.float64)
    for s, e in _chunks():
        R0 = R[s:e].toarray()
        M  = (R0 != 0).astype(np.float32)
        num_p = R0 @ St                                   # sum_j sim(i,j) * r_uj  (j rated by u)
        den_p = M  @ St                                   # sum_j sim(i,j)
        with np.errstate(invalid='ignore', divide='ignore'):
            pred = num_p / den_p
        pred[~np.isfinite(pred)] = global_mean
        completed = np.where(M > 0, R0, pred).astype(np.float32)
        G      += completed.T @ completed
        colsum += completed.sum(axis=0)
    del St

    # --- Final cosine of the completed, item-mean-centered ratings: continuous [0, 1].
    tmp = np.outer(colsum, colsum); tmp /= n_users
    G -= tmp; del tmp
    norm = np.sqrt(np.clip(np.diag(G), 0.0, None))
    on = np.outer(norm, norm)
    with np.errstate(invalid='ignore', divide='ignore'):
        np.divide(G, on, out=G)
    del on
    G[~np.isfinite(G)] = 0.0
    np.clip(G, 0.0, 1.0, out=G)                           # keep similarity in [0, 1]
    u = G.astype(np.float32)
    del G
    np.fill_diagonal(u, 0.0)
    if verbose:
        denom_off = float(N * N - N)
        print("U built: %dx%d float32 | users=%d | mean=%.4f max=%.4f frac>0.7=%.3f%%"
              % (N, N, n_users, u.sum() / denom_off, float(u.max()),
                 100.0 * int((u > 0.7).sum()) / denom_off))
    return u


def create_popularity(number_of_contents, dataset=None, top_movies=None, df=None):

    if dataset is None and df is None:
        zipf_s = 1.0
        _ranks = np.arange(1, number_of_contents + 1)
        zipf_pop = 1.0 / np.power(_ranks, zipf_s)
        popularity_probabilities = (zipf_pop / zipf_pop.sum()).astype(np.float32)
        np.random.shuffle(popularity_probabilities)
    else:
        if df is None:
            df = pd.read_csv(dataset)
        df = normalize_ratings_schema(df)
        if 'timestamp' in df.columns:
            df = df.drop('timestamp', axis='columns')

        if top_movies is None:
            top_movies = get_top_movies(number_of_contents, df=df)

        df_filtered = df[df['movieId'].isin(top_movies)]
        rating_counts = df_filtered.groupby('movieId')['userId'].count()

        # Build probabilities in the SAME order as top_movies
        counts = np.array([rating_counts.get(m, 0) for m in top_movies], dtype=np.float32)
        popularity_probabilities = np.array(counts / counts.sum())

    return popularity_probabilities


def find_action_index_from_states(cache_states,state_id,next_state_id,cache_size,id_to_cache):
    state_info=find_state_by_id_faster(cache_states,id_to_cache,state_id)
    next_state_info=find_state_by_id_faster(cache_states,id_to_cache,next_state_id)
    index=0
    for i in state_info[1]:
        if i not in next_state_info[1]:
            break
        index+=1
    else:
        index=cache_size
    # index of action in q-table
    return (next_state_info[0]*(cache_size+1)) +index


def action_format_pi(cache_size,current_state,action_index):
    cache_action= action_index % (cache_size+1)
    content_action = action_index // (cache_size+1)
    state=content_action,tuple(generate_cache_state(cache_size,current_state,cache_action))

    return state

def action_format(cache_size,current_state,action_index,topN):
    cache_action= action_index % (cache_size+1)
    content_action_index = action_index // (cache_size+1)
    content_action=topN[content_action_index]

    state=content_action,generate_cache_state(cache_size,current_state,cache_action)

    return state




def find_all_states(number_of_contents,cache_size):

    # all possible cache states
    cached_combinations = list(combinations(range(0, number_of_contents), cache_size))

    #dictionary whit all possible states(current + cache)
    state_space = {}
    state_index = 0
    for content in range(0, number_of_contents):
        for cached_contents in cached_combinations:
            state_space[(content, cached_contents)] = state_index
            state_index += 1

    return   state_space

def find_all_cache_states(number_of_contents,cache_size):

    # all possible cache states
    cached_combinations = list(combinations(range(0, number_of_contents), cache_size))

    cache_state = {}
    cache_index = 0

    for cache_contents in cached_combinations:
        cache_state[cache_contents] = cache_index
        cache_index+=1

    return cache_state


def find_state_by_id_faster(cache_states,id_to_cache,state_id):
    n = len(cache_states)
    content = state_id // n
    cache = id_to_cache[state_id % n]
    return content, cache


def find_action_index_from_states_DQN(cache_size,state,next_state,topN):
    index=0
    for i in state[1]:
        if i not in next_state[1]:
            break
        index+=1
    else:
        index=cache_size

    next_state_index=topN.index(next_state[0].item())
    return (next_state_index*(cache_size+1)) +index



def check_cache(cache1,cache2):
   # Checks if the cache difference is less than or equal to one content.
     return len(set(cache1) - set(cache2)) <= 1


def check_action_PI(cache_states,cache_size,current_state_id,next_state_id,action_idx,id_to_cache,):
    cache_action= action_idx % (cache_size+1)
    current_state=find_state_by_id_faster(cache_states,id_to_cache,current_state_id)
    next_state=find_state_by_id_faster(cache_states,id_to_cache,next_state_id)
    if(next_state[0]==current_state[0]):
      return False
    else:
      if(current_state[1]==next_state[1] and cache_action==cache_size): # not save same cache
          return True
      elif(check_cache(current_state[1],next_state[1]) and current_state[1][cache_action] not in next_state[1]  and current_state[0] in next_state[1] ): # save, max cache difference one content (the currelnty watched)
          return True
      return False



def check_action(cache_states,current_state_id,id_to_cache,next_state_id):
   # Returns True if eligible action, else False.
   current_state=find_state_by_id_faster(cache_states,id_to_cache,current_state_id)
   next_state=find_state_by_id_faster(cache_states,id_to_cache,next_state_id)
   if(next_state[0]==current_state[0]):
      return False
   else:
        if(current_state[1]==next_state[1]): # not save same cache
            return True
        elif(check_cache(current_state[1],next_state[1]) and current_state[0] in next_state[1] ): # save, max cache difference one content (the currelnty watched)
            return True
        return False

def check_action_DQN(current_state,next_state):
    """
    Returns True if eligible action, else False.
    """
    if(next_state[0]==current_state[0]):
        return False
    else:
        if(current_state[1]==next_state[1]): # not save same cache
            return True
        elif(check_cache(current_state[1],next_state[1]) and current_state[0] in next_state[1] ): # save, max cache difference one content (the currelnty watched)
            return True
        return False

def generate_random_cache_state(cache_size, current_state):
    """
    Creates an eligible random state of the cache.
    """
    
    cache = list(current_state[1])
    index_to_change = random.randint(-1, cache_size-1)
    if(index_to_change!=-1 and current_state[0] not in cache):
        cache[index_to_change] = current_state[0]
    return cache

def generate_cache_state(cache_size, current_state,index_to_change):
    # Creates an eligible random state of the cache
    
    cache = list(current_state[1])
    if(index_to_change!=cache_size and current_state[0] not in cache):
        cache[index_to_change] = current_state[0]
    return cache



def find_state_by_id(states,id):

    for state, index in states.items():
        if(index==id):
           return state
    raise ValueError("State not Found , id given : ",id, ", Number of states : ",states.__len__())

def generate_random_state(number_of_contents,cache_size):

    state_id = random.randint(0, number_of_contents-1)

    # Generate a random cache of size cache_size
    cache = random.sample(range(number_of_contents), cache_size)

    # Construct the state as a tuple of state_id and cache
    state = state_id,cache

    return state

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    cudnn.deterministic=True
    np.random.seed(seed)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

## Plot Utility

In [ ]:

def graphic_compare_results_reward(list_of_results,eps_decay,title=None,label=None,episode_slices=1000,file_name=None):

    for j in range(len(list_of_results)):
        reward_list=[]
        for i in range(list_of_results[j].max_iter//episode_slices):
            reward_list.append(100*np.mean(list_of_results[j].reward[i*episode_slices:(i+1)*episode_slices]))
        # reward_list_100=[i*100 for i in reward_list]
        if(label is not None):
            plt.plot([i*episode_slices for i in range(int(list_of_results[j].max_iter/episode_slices))],reward_list,label=label[j])
        else:
            plt.plot([i*episode_slices for i in range(int(list_of_results[j].max_iter/episode_slices))],reward_list,label=str(j+1))

    if title is not None:
        plt.title(title)
    else:
        plt.title("Cache hit rate MACRO (per-episode hits/steps)")
    plt.legend()
    plt.xlabel("Episodes")
    plt.ylabel("Cache hit rate per episode %")
    plt.ylim((0,100))
    plt.grid()
    if file_name is not None:
        plt.savefig(file_name)
    plt.show()




def graphic_compare_results_cost(list_of_results,eps_decay,title=None,label=None,episode_slices=1000,file_name=None):

    for j in range(len(list_of_results)):
        costs_list=[]
        for i in range(list_of_results[j].max_iter//episode_slices):
            costs_list.append(np.mean(list_of_results[j].cost[i*episode_slices:(i+1)*episode_slices]))
        if(label is not None):
            plt.plot([i*episode_slices for i in range(int(list_of_results[j].max_iter/episode_slices))],costs_list,label=label[j])
        else:
            plt.plot([i*episode_slices for i in range(int(list_of_results[j].max_iter/episode_slices))],costs_list,label=str(j+1))

    if title is not None:
        plt.title(title)
    else:
        plt.title("Q-value Update Difference")
    plt.legend()
    plt.xlabel("Episodes")
    plt.ylabel("MSE Loss")
    # plt.ylim((0,1))
    plt.grid()
    if file_name is not None:
        plt.savefig(file_name)
    plt.show()





def reward_plotting(list_of_rewards,episodes,title=None,label=None,episode_slices=1000):

    reward_list=[]
    for i in range(len(list_of_rewards)//episode_slices):
        reward_list.append(np.mean(list_of_rewards[i*episode_slices:(i+1)*episode_slices]))
    if(label is not None):
        plt.plot([i*episode_slices for i in range(len(list_of_rewards)//episode_slices)],reward_list,label=label[0])
    else:
        plt.plot([i*episode_slices for i in range(len(list_of_rewards)//episode_slices)],reward_list,label=str(0))

    if title is not None:
        plt.title(title)
    else:
        plt.title("Reward")
    plt.legend()
    plt.xlabel("Episodes")
    plt.ylabel("CHR %")
    plt.grid()
    plt.show()

## Training Utility

In [ ]:
from IPython.display import display

def _periodic_plot(episodes, plot_interval, rewards, costs, extra_lists, extra_titles,
                   episode_slices, threshold, early_stopage=False, plot_handle=None):
    try:
        n_plots = 1 + bool(costs) + (len(extra_lists) if extra_lists else 0)
        fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 4))

        if n_plots == 1:
            axes = [axes]

        ax_idx = 0

        # ── shared x-axis helper ──────────────────────────────────────────
        def make_x(lst):
            n = len(lst) // episode_slices
            return [(i + 1) * episode_slices for i in range(n)]  # e.g. 500, 1000, 1500 …

        # Cache Hit Rate
        reward_list = [
            100 * np.mean(rewards[i * episode_slices:(i + 1) * episode_slices])
            for i in range(len(rewards) // episode_slices)
        ]
        axes[ax_idx].plot(make_x(rewards), reward_list)
        axes[ax_idx].set_title("Cache Hit Rate")
        axes[ax_idx].set_xlabel("Episode")
        axes[ax_idx].set_ylabel("Cache hit rate per episode %")
        axes[ax_idx].set_ylim(0, 100)
        axes[ax_idx].grid(True)
        ax_idx += 1

        # Extra lists
        if extra_lists:
            for lst, title in zip(extra_lists, extra_titles):
                values = [
                    np.mean(lst[i * episode_slices:(i + 1) * episode_slices])
                    for i in range(len(lst) // episode_slices)
                ]
                axes[ax_idx].plot(make_x(lst), values)
                axes[ax_idx].set_title(title)
                axes[ax_idx].set_xlabel("Episode")
                axes[ax_idx].grid(True)
                ax_idx += 1

        # Loss
        if costs:
            cost_list = [
                np.mean(costs[i * episode_slices:(i + 1) * episode_slices])
                for i in range(len(costs) // episode_slices)
            ]
            axes[ax_idx].plot(make_x(costs), cost_list)
            axes[ax_idx].set_title("Loss")
            axes[ax_idx].set_xlabel("Episode")
            axes[ax_idx].set_ylabel("MSE Loss")
            axes[ax_idx].grid(True)

        plt.suptitle(f"Episode {episodes}", fontsize=12)
        plt.tight_layout()

        if plot_handle is not None:
            plot_handle.update(fig)
        else:
            plt.show()

        plt.close(fig)

    except Exception:
        pass

    if threshold > 0 and episodes >= 2 * plot_interval and early_stopage == True:
        delta = np.abs(np.mean(rewards[-plot_interval:]) - np.mean(rewards[-2 * plot_interval:-plot_interval]))
        if delta < threshold:
            print(f"Converged (Δ={delta:.6f} < {threshold}). Stopping")
            return True

    return False

## Cleanup Utility

In [ ]:
def finalize_training(agent):
    """Call this once training is complete. Keeps the agent fully usable for inference."""

    # 1. Experience replay buffer — largest single allocation, never needed after training
    del agent.memory
    agent.memory = None

    # 2. Target network — only used during learning, not during choose_actions
    del agent.q_network_target
    agent.q_network_target = None

    # 3. Optimizer and scheduler — only needed for gradient updates
    del agent.optimizer
    del agent.scheduler
    agent.optimizer = None
    agent.scheduler = None

    # 4. Gradient scaler — only used for mixed precision training
    del agent.scaler
    agent.scaler = None


    # 5. Force Python garbage collector to reclaim freed memory immediately
    import gc
    gc.collect()

    # 6. Release any unused GPU memory back to the allocator
    if agent.device.type == "cuda":
        torch.cuda.empty_cache()

    # 7. Lock the policy network into eval mode permanently
    agent.q_network_policy.eval()
    for param in agent.q_network_policy.parameters():
        param.requires_grad = False

    print("Training cleanup complete.")

# Environemnt

In [ ]:
# this Environment class replaces the 5-6 separate env classes I had before
# (Environment_PI, Environment_without_caching, Environment_with_caching, Environment_DQN_NC/WC, Non_RL_env)
# they were all basically the same thing with small differences, so I merged them.
#
# tabular=True  -> builds the full state table, used for Policy Iteration / Q-learning (only works for small N)
# tabular=False -> tuple states (content_id, cache_list), used for DQN / non-RL agents, scales to N=5000 etc.
#
# note: cache is always kept as a plain list (not tuple) in tuple mode so agent code can do state[1]+[x] etc.

# Environment_PI = Environment                 # tabular=True
# Environment_without_caching = Environment    # tabular=True
# Environment_with_caching = Environment       # tabular=True
# Environment_DQN_NC = Environment             # tabular=False (default)
# Environment_DQN_WC = Environment             # tabular=False (default)
# Non_RL_env = Environment                     # tabular=False (default)


def _choose_random_state(popularity):
    return np.random.choice(len(popularity), p=popularity)
    


def _check_cache(cache1, cache2):
    # true if the two caches differ by at most one item
    return sum(1 for c in cache1 if c not in cache2) <= 1


def _check_action_tabular(cache_states, cache_size, cur_id, nxt_id, id_to_cache):
    current_state=_state_by_id(cache_states,id_to_cache,cur_id)
    next_state=_state_by_id(cache_states,id_to_cache,nxt_id)
    if(next_state[0]==current_state[0]):
        return False
    else:
          if(current_state[1]==next_state[1]): # not save same cache
              return True
          elif(_check_cache(current_state[1],next_state[1]) and current_state[0] in next_state[1] ): # save, max cache difference one content (the currelnty watched)
              return True
          return False




def _state_by_id(cache_states, id_to_cache,state_id):
    # decode integer state id back into (content, cache_tuple)
    n = len(cache_states)
    return state_id // n, id_to_cache[state_id % n]



def _build_all_states(number_of_contents, cache_size):
    # builds {(content, cache_tuple): state_id} and {cache_tuple: cache_id}, only used in tabular mode
    combos = list(combinations(range(number_of_contents), cache_size))
    cache_states = {c: i for i, c in enumerate(combos)}
    all_states = {}
    sid = 0
    for content in range(number_of_contents):
        for cache in combos:
            all_states[(content, cache)] = sid
            sid += 1
    return all_states, cache_states


def _random_tuple_state(number_of_contents, cache_size):
    content = random.randrange(number_of_contents)
    cache = random.sample(range(number_of_contents), cache_size)  # list, not tuple
    return content, cache


class Environment:
    # number_of_contents, cache_size: catalogue / cache sizes
    # rewards: [miss_reward, hit_reward]
    # prob_follow: threshold/probability the user follows a recommendation
    # prob_leave: probability the user leaves the session each step
    # user_type: 'random' or 'quality_aware'
    # dataset: path to ratings csv, if None u/popularity are generated randomly
    # u, popularity: precomputed similarity matrix / popularity vector, if you already have them
    # tabular: True enumerates the whole state space (needed for PI/Q-learning), False is the DQN/non-RL mode (default)

    def __init__(
        self,
        number_of_contents,
        cache_size,
        rewards,
        number_of_recommendations,
        prob_follow,
        prob_leave,
        user_type,
        dataset=None,
        u=None,
        popularity=None,
        tabular=False,
    ):
        self.number_of_contents      = number_of_contents
        self.cache_size              = cache_size
        self.rewards                 = rewards
        self.number_of_recommendations = number_of_recommendations
        self.prob_follow             = prob_follow
        self.probability_to_follow_recommendation = prob_follow  # alias, some old loops still use this name
        self.prob_leave              = prob_leave
        self.user_left               = prob_leave    # alias, some old loops still use this name
        self.user_type               = user_type
        self.tabular                 = tabular

        # similarity matrix
        if u is not None:
            self.u = u
        else:
            self.u=create_u(number_of_contents,dataset)


        # popularity
        if popularity is not None:
            self.popularity = popularity
        else:
            self.popularity = create_popularity(number_of_contents,dataset)

        # popularity with the current content zeroed out and renormalized, precomputed per content
        self._pop_excluding = []
        for c in range(number_of_contents):
            p = self.popularity.copy().astype(np.float64)
            p[c] = 0.0
            p /= p.sum()
            self._pop_excluding.append(p)

        # precompute which contents are "good enough" recommendations for each content
        self.recommended = [
            np.where(self.u[c] >= self.prob_follow)[0]
            for c in range(number_of_contents)
        ]

        # only needed for tabular mode
        if tabular:
            self.all_states, self.cache_states = _build_all_states(
                number_of_contents, cache_size
            )
            self.number_of_states  = len(self.all_states)
            self.number_of_actions = self.number_of_states
            self.id_to_cache       = { idx: cache for cache,idx in self.cache_states.items()}
            self.reward_matrix     = self._build_reward_matrix()


    def _build_reward_matrix(self):
        # reward_matrix[state_id][content] = 1 if content is cached (and isn't the one being watched), else 0
        # tabular mode only
        matrix = np.zeros((self.number_of_states, self.number_of_contents), dtype=np.float32)
        for sid in range(self.number_of_states):
            content, cache = _state_by_id(self.cache_states,self.id_to_cache, sid)
            for cached_item in cache:
                if cached_item != content:
                    matrix[sid][cached_item] = 1.0
        return matrix

    def _load_u_from_dataset(self, dataset):
        # builds the similarity matrix from a ratings csv using cosine similarity
        df = pd.read_csv(dataset)
        if 'timestamp' in df.columns:
            df = df.drop('timestamp', axis=1)
        top_items = (
            df.groupby('movieId')['userId']
            .count()
            .sort_values(ascending=False)
            .head(self.number_of_contents)
            .index
        )
        df = df[df['movieId'].isin(top_items)]
        rating_matrix = df.pivot_table(index='userId', columns='movieId', values='rating').fillna(0)
        sparse = csr_matrix(rating_matrix)
        u = cosine_similarity(sparse).astype(np.float32)
        return u



    def is_cached(self, cache, content):
        return content in cache

    def calculate_reward(self, current_state, next_state):
        # works for both modes: tabular states are ints, tuple states are (content, cache_tuple)
        if current_state is None or next_state is None:
            return None

        if self.tabular:
            _, cache = _state_by_id(self.cache_states,self.id_to_cache, current_state)
            next_content, _ = _state_by_id(self.cache_states,self.id_to_cache, next_state)
        else:
            cache = current_state[1]
            next_content = next_state[0]

        return self.rewards[1] if self.is_cached(cache, next_content) else self.rewards[0]

    def calculate_reward_faster(self, current_state, next_state):
        # matrix lookup version, tabular mode only, kept for the old training loops
        if not self.tabular:
            raise RuntimeError("calculate_reward_faster is only available in tabular mode.")
        next_content = _state_by_id(self.cache_states,self.id_to_cache, next_state)[0]
        return self.reward_matrix[current_state][next_content]

    def simulate(self, action, state):
        # simulates one user step and returns new_state, reward, done
        # action is a single-element list: [int] in tabular mode, [(content_id, cache_tuple)] in tuple mode
        done = random.random() < self.prob_leave

        if self.tabular:
            return self._simulate_tabular(action, state, done)
        else:
            return self._simulate_tuple(action, state, done)

    def _simulate_tabular(self, action, state, done):
        # tabular version, states are integer ids
        action_info = _state_by_id(self.cache_states,self.id_to_cache, random.choice(action))
        state_info  = _state_by_id(self.cache_states,self.id_to_cache,state)

        if np.mean(self.u[state_info[0], action_info[0]]) > self.prob_follow:
            new_state = random.choice(action)
        else:
            new_state_info = (
                _choose_random_state(self.popularity),
                action_info[1],
            )
            new_state = self.all_states[new_state_info]
            while not _check_action_tabular(
                self.cache_states, self.cache_size,
                state, new_state,
                self.id_to_cache
            ):
                new_state_info = (
                    _choose_random_state(self.popularity),
                    action_info[1],
                )
                new_state = self.all_states[new_state_info]

        reward = self.calculate_reward_faster(state, new_state)
        return new_state, reward, done

    def _simulate_tuple(self, action, state, done):
        # tuple version, states are (content_id, list[int])
        # cache is always turned into a list here regardless of what the agent passed in
        action_to_take = random.choice(action)
        cache = list(action_to_take[1])

        if self.user_type == 'random':
            if (random.random() < self.prob_follow
                    and action_to_take[0] != state[0]):
                new_state = action_to_take[0], cache
            else:
                new_state = int(np.random.choice(self.number_of_contents, p=self._pop_excluding[state[0]])),cache

        elif self.user_type == 'quality_aware':
            if np.mean(self.u[state[0], action_to_take[0]]) > self.prob_follow:
                new_state = action_to_take[0], cache
            else:
               new_state = int(np.random.choice(self.number_of_contents, p=self._pop_excluding[state[0]])),cache

        else:
            raise ValueError(f"Unknown user_type '{self.user_type}'. "
                             "Use 'random' or 'quality_aware'.")

        reward = self.calculate_reward(state, new_state)

        return new_state, reward, done

    def refresh(self):
        # returns a random starting state: int in tabular mode, (content_id, cache_tuple) otherwise
        if self.tabular:
            return random.randrange(self.number_of_states)
        return _random_tuple_state(self.number_of_contents, self.cache_size)


# Policy Iteration

## Policy Iteration Agent

In [ ]:

class Agent_PI:
    def __init__(self,env,number_of_contents,cache_size,probability_to_follow_recommendation,u=None,dataset=None,popularity=None,gamma=0.95,max_iter=5000,threshold=1e-4):

        self.gamma=gamma
        self.number_of_contents=number_of_contents
        self.cache_size=cache_size
        self.threshold=threshold
        self.env=env
        if u is None:
            self.u=create_u(self.number_of_contents)
        else:
            self.u=u

        if popularity is None:

            self.popularity=create_popularity(self.number_of_contents,dataset)
        else:
            self.popularity=popularity

        self.all_states=find_all_states(self.number_of_contents,self.cache_size)
        self.cache_states=find_all_cache_states(self.number_of_contents,self.cache_size)
        self.number_of_states=self.all_states.__len__()
        self.number_of_actions=self.number_of_contents*(self.cache_size+1)
        self.probability_to_follow_recommendation=probability_to_follow_recommendation
        self.max_iter=max_iter
        self.reward_matrix=self.create_reward_matrix()
        self.transition_matrix=self.create_transition_matrix()

        self.policy,self.value=self.PI_train()


    def create_reward_matrix(self):
        reward_matrix=np.zeros([self.number_of_states,self.number_of_contents])
        for i in range(len(self.all_states)):
            state=find_state_by_id_faster(self.cache_states,self.env.id_to_cache,i)
            for j in state[1]:
                reward_matrix[i][j]=1
                reward_matrix[i][state[0]]=0
        return reward_matrix

    def create_transition_matrix(self):
        transition_matrix=np.empty((self.number_of_states,self.number_of_actions),dtype=object)
        for i in range(len(self.all_states)):
            state=find_state_by_id_faster(self.cache_states,self.env.id_to_cache,i)
            for j in range(self.number_of_actions):
                next_state_list=[]
                next_state=action_format_pi(self.cache_size,state,j)
                next_state_id=self.all_states[next_state]
                if(check_action_PI(self.cache_states,self.cache_size,i,next_state_id,j,self.env.id_to_cache)==True):
                    if( self.u[state[0]][next_state[0]]>self.probability_to_follow_recommendation):
                        next_state_list.append(next_state_id)
                        transition_matrix[i][j] = [[1.0] , [self.reward_matrix[i][next_state[0]]] ,next_state_list]
                    else:
                        probs=[]
                        rewards=[]
                        for k in range(self.number_of_states):
                            possible_state=find_state_by_id_faster(self.cache_states,self.env.id_to_cache,k)
                            if  state[0]!=possible_state[0] and next_state[1]==possible_state[1]:
                               next_state_list.append(k)
                               probs.append(self.popularity[possible_state[0]])
                               rewards.append(self.reward_matrix[i][possible_state[0]])


                        transition_matrix[i][j]=[probs,rewards,next_state_list]

                else:
                    continue

        return transition_matrix




    def PI_train(self):

        policy=np.random.choice(self.number_of_actions,self.number_of_states)

        while True:

            # policy evaluation
            V=np.zeros(self.number_of_states)
            for _ in range(self.max_iter):
                delta=0
                for s in range(self.number_of_states):
                    v=V[s]
                    action=policy[s]
                    if(self.transition_matrix[s][action] is None):
                        continue
                    prob,reward,next_state=self.transition_matrix[s][action]
                    V[s]= sum(prob[i] * (reward[i] + self.gamma * V[next_state[i]]) for i in range(len(prob)))

                    delta=max(delta,abs(v-V[s]))
                if(delta < self.threshold):
                    break

            # policy improvement
            policy_stable=True
            for s in range(self.number_of_states):
                old_action=policy[s]
                action_values=np.zeros(self.number_of_actions)
                for action in range(self.number_of_actions):
                    if(self.transition_matrix[s][action] is None):
                        continue
                    prob,reward,next_state=self.transition_matrix[s][action]
                    action_values[action]= sum(prob[i] * (reward[i] + self.gamma * V[next_state[i]]) for i in range(len(prob)))
                policy[s]=np.argmax(action_values)
                if old_action != policy[s]:
                    policy_stable=False
            if policy_stable:
                break

        print("Optimal Policy:",policy)
        print("optimal Value Function",V)

        return policy,V

    def PI_run(self,state_id):
        act=[]
        action=self.policy[state_id]
        state_info=find_state_by_id_faster(self.cache_states,self.env.id_to_cache,state_id)
        next_state_info=action_format_pi(self.cache_size,state_info,action)
        next_state_id=self.all_states[next_state_info]

        act.append(next_state_id)

        return act

## Policy Iteration Update Function

In [ ]:
def update_PI(env,agent,max_iter = 1000,plot_interval=2000,threshold = 0.1):


    episodes=0
    max_difference=threshold+1
    start_time = time.time()

    list_of_rewards = []

    while ( episodes < max_iter):


        state = env.refresh()
        done = False
        i=0
        current_penalty = 0
        current_reward = 0

        while (not done):
            action=agent.PI_run(state)

            next_state,reward,done=env.simulate(action,state)

            state=next_state

            i+=1
            current_reward+=reward

            if reward == env.rewards[0]:
                current_penalty+=1

        episodes+=1

        if(i!=0):
            list_of_rewards.append(current_reward/i)

        if(episodes % plot_interval == 0):
            print('Episodes : ',episodes, ' / ' , max_iter)
            _periodic_plot(episodes,plot_interval,list_of_rewards,costs=None,
                           extra_lists=None, extra_titles=None, episode_slices=500,
                           threshold=threshold,early_stopage=False)



    running_time=time.time()-start_time

    return running_time,episodes,list_of_rewards

# Q-Learning

## Q-Learning without caching decisions

### Q-Learning Agent without caching decisions

In [ ]:
class Agent_without_caching:
# The agent class simulates the system
    def __init__(self,env,learning_rate=0.1,gamma=0.8,eps=0.9,max_iter=50000):

        self.learning_rate=learning_rate
        self.gamma=gamma
        self.number_of_states=env.number_of_states
        self.all_states=env.all_states
        self.cache_states=env.cache_states
        self.number_of_contents=env.number_of_contents
        self.cache_size=env.cache_size
        self.env=env

        self.q_table=np.zeros([self.number_of_states,self.number_of_contents])

        # E-greedy parameters
        self.eps_init=eps
        self.eps=self.eps_init
        self.eps_decay_rate=5*1e-5
        self.eps_min=0.05

    def choose_actions(self,state):
        action=[]

        if random.uniform(0,1) < self.eps:
            #Explore state space
            state_info=find_state_by_id_faster(self.cache_states,self.env.id_to_cache,state)
            next_state_info=random.randint(0,self.number_of_contents-1),tuple(state_info[1])
            next_state=self.all_states[next_state_info]
            while(state_info[0]==next_state_info[0]):
                next_state_info=random.randint(0,self.number_of_contents-1),tuple(state_info[1])
                next_state=self.all_states[next_state_info]
        else:
            #Exploit learned values
            i=1
            state_info=find_state_by_id_faster(self.cache_states,self.env.id_to_cache,state)
            top_k = np.argpartition(self.q_table[state], -self.number_of_contents)[-10:]
            action_index = top_k[np.argsort(self.q_table[state][top_k])[-i]]
            next_state_info=action_index,tuple(state_info[1])
            next_state=self.all_states[next_state_info]
            while(state_info[0]==next_state_info[0]):
                i+=1
                action_index=np.argsort(self.q_table[state])[-i]
                next_state_info=action_index,tuple(state_info[1])
                next_state=self.all_states[next_state_info]

          # baseline policy(caching most popular)

        if self.env.popularity[state_info[0]]>np.min(self.env.popularity[list(state_info[1])]) and state_info[0] not in state_info[1]:
            index=np.argmin(self.env.popularity[list(state_info[1])])
            cache=generate_cache_state(self.env.cache_size,state_info,index)
            next_state_info=next_state_info[0],tuple(cache)
            next_state=self.all_states[next_state_info]

        action.append(next_state)

        return action


    def learn(self,state,action,reward,next_state):
        new_value=0
        old_value=0
        for act in action:
        # TO DO
            action_index=find_state_by_id_faster(self.cache_states,self.env.id_to_cache,act)[0]
            old_value=self.q_table[state,action_index]
            next_state_max=np.max(self.q_table[next_state])
            new_value=(1-self.learning_rate)*old_value + self.learning_rate * ( reward + self.gamma * next_state_max)
            self.q_table[state,action_index] = new_value
        return new_value,old_value

    def eps_decay(self,episode):
        self.eps=self.eps_min +(self.eps_init-self.eps_min)*math.exp(-episode*self.eps_decay_rate)
        return

## Q-Learning with caching decisions

### Q-Learning Agent capable of making caching decisions

In [ ]:
class Agent_with_caching:
# The agent class simulates the system that takes the reccomendations and caching decisions
    def __init__(self,env,learning_rate=0.1,gamma=0.8,eps=0.9,max_iter=50000):
        self.env=env
        self.learning_rate=learning_rate
        self.gamma=gamma
        self.number_of_states=self.env.number_of_states
        self.all_states=self.env.all_states
        self.cache_states=self.env.cache_states
        self.number_of_contents=self.env.number_of_contents
        self.cache_size=self.env.cache_size

        self.q_table=np.zeros([self.number_of_states,self.number_of_contents*(self.cache_size+1)])

        # E-greedy parameters
        self.eps_init=eps
        self.eps=self.eps_init
        self.eps_decay_rate=5*1e-5
        self.eps_min=0.05

    def choose_actions(self,state):
        action=[]

        if random.uniform(0,1) < self.eps:
            #Explore state space
            state_info=find_state_by_id_faster(self.cache_states,self.env.id_to_cache,state)
            next_state_info=(random.randint(0,self.number_of_contents-1),tuple(generate_random_cache_state(self.cache_size,state_info)))
            next_state=self.all_states[next_state_info]
            while(check_action(self.cache_states,state,self.env.id_to_cache,next_state)== False):
                next_state_info=(random.randint(0,self.number_of_contents-1),tuple(generate_random_cache_state(self.cache_size,state_info)))
                next_state=self.all_states[next_state_info]

            action.append(next_state)
        else:
            #Exploit learned values
            i=1
            state_info=find_state_by_id_faster(self.cache_states,self.env.id_to_cache,state)
            top_k = np.argpartition(self.q_table[state], -self.number_of_contents)[-10:]
            action_index = top_k[np.argsort(self.q_table[state][top_k])[-i]]
            next_state_info=action_format_pi(self.cache_size,state_info,action_index)
            next_state=self.all_states[next_state_info]
            while(check_action(self.cache_states,state,self.env.id_to_cache,next_state)==False):
                i+=1
                action_index=np.argsort(self.q_table[state])[-i]
                next_state_info=action_format_pi(self.cache_size,state_info,action_index)
                next_state=self.all_states[next_state_info]


            action.append(next_state)
        return action


    def learn(self,state,action,reward,next_state):
        new_value=0
        old_value=0
        for act in action:
            action_index=find_action_index_from_states(self.cache_states,state,act,self.cache_size,self.env.id_to_cache)
            old_value=self.q_table[state,action_index]
            next_state_max=np.max(self.q_table[next_state])
            new_value=(1-self.learning_rate)*old_value + self.learning_rate * ( reward + self.gamma * next_state_max)
            self.q_table[state,action_index] = new_value
        return new_value,old_value


    def eps_decay(self,episode):
        self.eps=self.eps_min +(self.eps_init-self.eps_min)*math.exp(-episode*self.eps_decay_rate)
        return

## Q-Learning Update Function

In [ ]:

def update(env,agent,plot_handle,max_iter = 1000,plot_interval=1000,threshold = 0.1,eps_decay=False,train_enable=True):

    costs = []
    episodes=0
    counter=0
    max_difference=threshold+1
    start_time = time.time()
    total_steps=0
    total_hits=0

    list_of_rewards = []
    if( not train_enable):
       eps_decay=False
       agent.eps=0


    for episodes in trange(max_iter):
        new_values=[]
        old_values=[]
        # old_q_table = copy.copy(agent.q_table)
        state = env.refresh()



        i = 0

        done = False

        current_penalty = 0
        current_reward = 0

        while (not done):
            action=agent.choose_actions(state)

            next_state,reward,done=env.simulate(action,state)


            if(train_enable):
                new_value,old_value=agent.learn(state,action,reward,next_state)
                new_values.append(new_value)
                old_values.append(old_value)




            state=next_state

            i+=1
            current_reward+=reward

            if reward == env.rewards[0]:
                current_penalty+=1

        episodes+=1



        if(train_enable):
          cost=np.square(np.subtract(new_values,old_values)).mean()
          costs.append(cost)

        if(i!=0):
            total_steps+=i
            total_hits+=current_reward
            list_of_rewards.append(total_steps/total_steps)

        if eps_decay:
            agent.eps_decay(episodes)

        if(episodes % plot_interval == 0):

            _periodic_plot(episodes,plot_interval,list_of_rewards,costs=costs,
                           extra_lists=None, extra_titles=None, episode_slices=500,
                           threshold=threshold,early_stopage=False,plot_handle=plot_handle)
            print(agent.eps)
    running_time=time.time()-start_time

    return running_time,episodes,costs,list_of_rewards

# DQN Model

In [ ]:
class NoisyLinear(nn.Module):
    """
    Noisy linear layer with factorized Gaussian noise for targeted exploration.
    """
    def __init__(self, in_features, out_features, sigma_init=0.5):
        super(NoisyLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.sigma_init = sigma_init

        # Learnable parameters for the mean (mu) and variance (sigma)
        self.weight_mu = nn.Parameter(torch.empty(out_features, in_features))
        self.weight_sigma = nn.Parameter(torch.empty(out_features, in_features))
        self.register_buffer('weight_epsilon', torch.empty(out_features, in_features))

        self.bias_mu = nn.Parameter(torch.empty(out_features))
        self.bias_sigma = nn.Parameter(torch.empty(out_features))
        self.register_buffer('bias_epsilon', torch.empty(out_features))

        self.reset_parameters()
        self.reset_noise()

    def reset_parameters(self):
        mu_range = 1 / math.sqrt(self.in_features)
        self.weight_mu.data.uniform_(-mu_range, mu_range)
        self.weight_sigma.data.fill_(self.sigma_init / math.sqrt(self.in_features))
        
        self.bias_mu.data.uniform_(-mu_range, mu_range)
        self.bias_sigma.data.fill_(self.sigma_init / math.sqrt(self.out_features))

    def _scale_noise(self, size):
        x = torch.randn(size)
        return x.sign().mul_(x.abs().sqrt_())

    def reset_noise(self):
        epsilon_in = self._scale_noise(self.in_features)
        epsilon_out = self._scale_noise(self.out_features)
        
        # Factorized Gaussian noise
        self.weight_epsilon.copy_(epsilon_out.ger(epsilon_in))
        self.bias_epsilon.copy_(epsilon_out)

    def forward(self, x):
        if self.training:
            # Inject noise during training
            weight = self.weight_mu + self.weight_sigma * self.weight_epsilon
            bias = self.bias_mu + self.bias_sigma * self.bias_epsilon
        else:
            # Evaluate using purely the learned mean during testing
            weight = self.weight_mu
            bias = self.bias_mu
            
        return F.linear(x, weight, bias)

## DQN Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class GatedLinearUnit(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.fc = nn.Linear(input_dim, output_dim * 2)

    def forward(self, x):
        value, gate = self.fc(x).chunk(2, dim=-1)
        return value * torch.sigmoid(gate)



class DQNFlex(nn.Module):
    def __init__(
        self,
        num_features,
        cache_size,
        num_contents,
        hidden_dim=128,
        latent_dim=32,
        rec_only=False
    ):
        super().__init__()

        self.K = cache_size
        self.num_contents = num_contents
        self.rec_only = rec_only

        # Encoder over raw per-candidate features (no item embedding)
        self.glu = GatedLinearUnit(num_features, hidden_dim)

        # Attention
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.cross_norm = nn.LayerNorm(hidden_dim)

        # No-eviction slot
        self.no_evict_emb = nn.Parameter(torch.randn(1, 1, hidden_dim))

        # Norms
        self.rec_norm = nn.LayerNorm(hidden_dim)
        self.cache_norm = nn.LayerNorm(hidden_dim)
        self.value_norm = nn.LayerNorm(hidden_dim)

        # Dueling heads
        self.value_head = nn.Sequential(NoisyLinear(hidden_dim, hidden_dim // 2), nn.ReLU(), NoisyLinear(hidden_dim // 2, 1))
        self.rec_head   = nn.Sequential(NoisyLinear(hidden_dim, hidden_dim // 2), nn.ReLU(), NoisyLinear(hidden_dim // 2, 1))
        self.cache_head = nn.Sequential(NoisyLinear(hidden_dim, hidden_dim // 2), nn.ReLU(), NoisyLinear(hidden_dim // 2, 1))

        # Bilinear synergy projections
        self.rec_proj = NoisyLinear(hidden_dim, latent_dim)
        self.cache_proj = NoisyLinear(hidden_dim, latent_dim)

    def reset_noise(self):
        for m in self.modules():
            if isinstance(m, NoisyLinear):
                m.reset_noise()

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(0)

        B, M, _ = x.shape

        # Encode candidates (raw features in)
        h = self.glu(x)

        # Attention
        query = h[:, 0:1, :]
        q = self.q_proj(query)
        k = self.k_proj(h)
        v = self.v_proj(h)
        context = F.scaled_dot_product_attention(q, k, v)
        contextual_query = self.cross_norm(query + context)
        h_contextual = h + contextual_query.expand(-1, M, -1)

        # State value stream
        global_state_vec = contextual_query.squeeze(1)
        V = self.value_head(self.value_norm(global_state_vec))
        V = V.unsqueeze(-1)  # [B, 1, 1]

        # Recommendation advantage stream
        A_rec = self.rec_head(self.rec_norm(h_contextual))
        A_rec = A_rec - A_rec.mean(dim=1, keepdim=True)  # [B, M, 1]

        if self.rec_only:
            Q = V + A_rec
            return Q.reshape(B, M)

        # Item-aware cache advantage stream
        h_cached = h_contextual[:, 1:self.K + 1, :]
        h_no_evict = h_cached.mean(dim=1, keepdim=True) 
        cache_state = self.cache_norm(torch.cat([h_cached, h_no_evict], dim=1))  # [B, K+1, H]

        A_cache = self.cache_head(cache_state)
        A_cache = A_cache - A_cache.mean(dim=1, keepdim=True)
        A_cache = A_cache.transpose(1, 2)  # [B, 1, K+1]

        # Bilinear synergy stream
        rec_latent = self.rec_proj(self.rec_norm(h_contextual))
        cache_latent = self.cache_proj(cache_state)

        A_int = torch.bmm(rec_latent, cache_latent.transpose(1, 2))
        row_mean = A_int.mean(dim=2, keepdim=True)   # [B, M, 1]
        col_mean = A_int.mean(dim=1, keepdim=True)   # [B, 1, K+1]
        grand_mean = A_int.mean(dim=(1, 2), keepdim=True)
        A_int = A_int - row_mean - col_mean + grand_mean

        # Fixed-sum aggregation (gates removed)
        Q_matrix = V + A_rec + A_cache + A_int

        return Q_matrix.reshape(B, M * (self.K + 1))

## N-Step

In [ ]:
from collections import deque

class NStepBuffer:
    """Accumulates n transitions, then yields the compressed n-step experience."""
    def __init__(self, n, gamma):
        self.n = n
        self.gamma = gamma
        self.buffer = deque()

    def append(self, exp: Experience):
        self.buffer.append(exp)

    def clear(self):
        self.buffer.clear()

    def is_ready(self):
        return len(self.buffer) >= self.n

    def get(self) -> Experience:
        """
        Returns one n-step Experience. The reward is the discounted sum over n steps,
        and next_state is the state n steps ahead.
        """
        n_step_reward = 0.0
        for i, exp in enumerate(list(self.buffer)[:self.n]):
            n_step_reward += (self.gamma ** i) * exp.reward
            if exp.done:  # truncate at episode end
                break

        first = self.buffer[0]
        last  = list(self.buffer)[min(self.n, len(self.buffer)) - 1]

        n_step_exp = Experience(
            state          = first.state,
            action         = first.action,
            reward         = n_step_reward,      # ← discounted n-step sum
            done           = last.done,           # ← done flag of the LAST step
            next_state     = last.next_state,     # ← state n steps ahead
            state_rep      = first.state_rep,
            next_state_rep = last.next_state_rep,
            state_topN     = first.state_topN,
            state_topN_map = first.state_topN_map,
            next_topN      = last.next_topN,
            next_topN_map  = last.next_topN_map,
        )
        self.buffer.popleft()
        return n_step_exp

## Experience Replay

In [ ]:

class ExperienceReplay:

    def __init__(self,capacity,device):
        self.capacity=capacity
        self.memory = collections.deque(maxlen=self.capacity)
        self.device = device
        self.pin_memory = self.device.type == "cuda"
        self.non_blocking = self.pin_memory

    def __len__(self):
        return len(self.memory)

    def append(self,experience):
        self.memory.append(experience)

    def sample(self, batch_size):
        # Randomly sample a batch of experiences from the replay buffer
        experiences = random.sample(self.memory, batch_size)

        rewards, dones, states_rep, next_states_rep = [], [], [], [],
        states_content,states_cache,next_states_content,next_states_cache=[],[],[],[]
        action_content,action_cache=[],[]
        state_topN,state_topN_map,next_topN,next_topN_map=[],[],[],[]
        for exp in experiences:
            states_content.append(exp.state[0])
            states_cache.append(exp.state[1])
            next_states_content.append(exp.next_state[0])
            next_states_cache.append(exp.next_state[1])
            action_content.append(exp.action[0][0])
            action_cache.append(exp.action[0][1])
            rewards.append(exp.reward)
            dones.append(exp.done)
            states_rep.append(exp.state_rep)
            next_states_rep.append(exp.next_state_rep)
            state_topN.append(exp.state_topN)
            state_topN_map.append(exp.state_topN_map)
            next_topN.append(exp.next_topN)
            next_topN_map.append(exp.next_topN_map)


        if isinstance(states_rep, list) and all(isinstance(s, torch.Tensor) for s in states_rep):
              states_batch_rep = torch.stack([s.to(self.device,non_blocking=self.non_blocking) for s in states_rep])

        if isinstance(next_states_rep, list) and all(isinstance(ns, torch.Tensor) for ns in next_states_rep):
              next_states_batch_rep = torch.stack([ns.to(self.device,non_blocking=self.non_blocking) for ns in next_states_rep])


        states_content = torch.as_tensor(states_content, dtype=torch.int32)
        states_cache = torch.as_tensor(states_cache, dtype=torch.int32)
        action_content = torch.as_tensor(action_content, dtype=torch.int32)
        action_cache = torch.as_tensor(action_cache, dtype=torch.int32)
        rewards = torch.as_tensor(rewards, dtype=torch.float32).to(device=self.device,non_blocking=True)
        dones = torch.as_tensor(dones, dtype=torch.int32).to(device=self.device,non_blocking=True)
        next_states_content = torch.as_tensor(next_states_content, dtype=torch.int32)#.to(device=self.device,non_blocking=True)
        next_states_cache = torch.as_tensor(next_states_cache, dtype=torch.int32)#.to(device=self.device,non_blocking=True)

        return (states_content,states_cache),(action_content,action_cache), rewards, dones, (next_states_content,next_states_cache), states_batch_rep, next_states_batch_rep,state_topN,state_topN_map,next_topN,next_topN_map

## Prioritized Experience Replay using Sum Tree

In [ ]:
class SumTree:

    def __init__(self,capacity):
        self.capacity= capacity
        self.tree=[0] * (2 * self.capacity - 1)
        self.data=[None] * self.capacity
        self.write_idx=0
        self.num_entries=0

    def total(self):
        return self.tree[0]

    def update(self,data_idx,priority):
        idx=data_idx+self.capacity-1
        difference=priority-self.tree[idx]
        self.tree[idx]=priority

        parent=(idx-1) // 2
        while parent>=0:
            self.tree[parent]+=difference
            parent=(parent-1) // 2


    def add(self,priority,data):
        self.data[self.write_idx]=data
        self.update(self.write_idx,priority)

        self.write_idx = (self.write_idx+1) % self.capacity
        self.num_entries = min(self.capacity,self.num_entries+1)

    def get(self,cumsum):
        cumsum = min(cumsum, self.total() - 1e-6)

        idx=0
        while 2*idx +1 < len(self.tree):
            left,right = 2*idx + 1  , 2*idx + 2

            if cumsum <=self.tree[left]:
                idx=left
            else:
                idx= right
                cumsum=cumsum - self.tree[left]

        data_idx=idx-self.capacity + 1

        return data_idx,self.tree[idx],self.data[data_idx]


class PrioritizedExperienceReplay:
    def __init__(self, capacity,device, eps=1e-2, alpha=0.7, beta=0.4):
        self.capacity = capacity
        self.tree = SumTree(capacity)
        self.memory = [None]*self.capacity
        self.alpha = alpha
        self.beta = beta
        self.max_priority = eps
        self.eps = eps
        self.max_beta = 1.0
        self.device = device
        self.pin_memory = self.device.type == "cuda"
        self.non_blocking = self.pin_memory


    def __len__(self):
        return self.tree.num_entries

    def append(self, experience):
        # Add experience to memory and SumTree
        self.memory[self.tree.write_idx] = experience
        self.tree.add(self.max_priority, self.tree.write_idx)

    def sample(self, batch_size):
        sample_idxs, tree_idxs = [], []
        priorities = np.empty([batch_size, 1], dtype=np.float32)
        segment = self.tree.total() / batch_size

        for i in range(batch_size):
            a, b = segment * i, segment * (i + 1)
            cumsum = random.uniform(a, b)
            tree_idx, priority, sample_idx = self.tree.get(cumsum)

            priorities[i] = priority
            tree_idxs.append(tree_idx)
            sample_idxs.append(sample_idx)

        probs = priorities / self.tree.total()
        probs   = np.clip(probs, 1e-8, None)
        weights = (self.tree.num_entries * probs) ** -self.beta
        weights = weights / weights.max()

        experiences = [self.memory[i] for i in sample_idxs]
        rewards, dones, states_rep, next_states_rep = [], [], [], [],
        states_content,states_cache,next_states_content,next_states_cache=[],[],[],[]
        action_content,action_cache=[],[]
        state_topN,state_topN_map,next_topN,next_topN_map=[],[],[],[]
        for exp in experiences:
            states_content.append(exp.state[0])
            states_cache.append(exp.state[1])
            next_states_content.append(exp.next_state[0])
            next_states_cache.append(exp.next_state[1])
            action_content.append(exp.action[0][0])
            action_cache.append(exp.action[0][1])
            rewards.append(exp.reward)
            dones.append(exp.done)
            states_rep.append(exp.state_rep)
            next_states_rep.append(exp.next_state_rep)
            state_topN.append(exp.state_topN)
            state_topN_map.append(exp.state_topN_map)
            next_topN.append(exp.next_topN)
            next_topN_map.append(exp.next_topN_map)


        if isinstance(states_rep, list) and all(isinstance(s, torch.Tensor) for s in states_rep):
              states_batch_rep = torch.stack([s.to(self.device,non_blocking=self.non_blocking) for s in states_rep])

        if isinstance(next_states_rep, list) and all(isinstance(ns, torch.Tensor) for ns in next_states_rep):
              next_states_batch_rep = torch.stack([ns.to(self.device,non_blocking=self.non_blocking) for ns in next_states_rep])


        states_content = torch.as_tensor(states_content, dtype=torch.int32)
        states_cache = torch.as_tensor(states_cache, dtype=torch.int32)
        action_content = torch.as_tensor(action_content, dtype=torch.int32)
        action_cache = torch.as_tensor(action_cache, dtype=torch.int32)
        rewards = torch.as_tensor(rewards, dtype=torch.float32).to(device=self.device,non_blocking=True)
        dones = torch.as_tensor(dones, dtype=torch.int32).to(device=self.device,non_blocking=True)
        next_states_content = torch.as_tensor(next_states_content, dtype=torch.int32)#.to(device=self.device,non_blocking=True)
        next_states_cache = torch.as_tensor(next_states_cache, dtype=torch.int32)#.to(device=self.device,non_blocking=True)
        weights = torch.as_tensor(weights, dtype=torch.float32).to(device=self.device,non_blocking=True)

        return (states_content,states_cache),(action_content,action_cache), rewards, dones, (next_states_content,next_states_cache), states_batch_rep, next_states_batch_rep, tree_idxs, weights,state_topN,state_topN_map,next_topN,next_topN_map

    def update_priority(self, indices, priorities):
        for data_idx, priority in zip(indices, priorities):
            priority = float(min(max(priority, self.eps), 1e6)) ** self.alpha
            # priority = priority ** self.alpha
            self.tree.update(data_idx, priority)
            self.max_priority = max(self.max_priority, priority)

    def beta_increase(self, increment):
        if self.beta + increment <= self.max_beta:
            self.beta += increment
        else:
            self.beta = self.max_beta

# Double DQN Agent

In [ ]:
class DQNAgent:

    def __init__(self,env,state_features=3,learning_rate=0.1,gamma=0.9,eps=0.9,max_iter=50000,replay_type="PER",replay_buffer_size=10000,batch_size=32,taf=0.01,rec_only=False):
            self.env=env
            self.learning_rate=learning_rate
            self.gamma=gamma
            self.max_iter=max_iter
            self.replay_type=replay_type
            self.replay_buffer_size=replay_buffer_size
            self.batch_size=batch_size
            self.taf=taf
            self.rec_only=rec_only

            self.state_features=state_features
            self.device=torch.device("cuda" if torch.cuda.is_available() else "cpu")


            # Recommended
            self.recommended_mask = torch.zeros(
                (self.env.number_of_contents,self.env.number_of_contents), dtype=torch.bool, device=self.device
            )

            for c, rec_set in enumerate(self.env.recommended):
                indices = torch.tensor(list(rec_set), dtype=torch.long)
                self.recommended_mask[c, indices] = True

            self.training_episodes=0
            

            self.rec_num=np.max(np.sum(self.env.u>self.env.prob_follow,axis=1))+self.env.cache_size
            self.rec_options_num=self.rec_num if self.rec_num< self.env.number_of_contents else self.env.number_of_contents

            self.checkpoint=f'{self.__class__}_{self.state_features}.chkpt'

            self.scaler = torch.amp.GradScaler(self.device.type)



            self.cache_loss=[]
            self.rec_loss=[]


            self.u_states=self.create_u_states()
            

        # Policy and Target networks
            self.q_network_policy=DQNFlex(self.state_features,self.env.cache_size,self.env.number_of_contents,rec_only=self.rec_only).to(self.device)
            self.q_network_target=DQNFlex(self.state_features,self.env.cache_size,self.env.number_of_contents,rec_only=self.rec_only).to(self.device)
            self.q_network_target.load_state_dict(self.q_network_policy.state_dict())
            self.q_network_target.eval()
            for param in self.q_network_target.parameters():
                param.requires_grad = False
        # Optimizer for the network
            decay_params = []
            no_decay_params = []

            for name, param in self.q_network_policy.named_parameters():
                if 'sigma' in name:
                    no_decay_params.append(param)
                else:
                    decay_params.append(param)
            
            # Pass groups to AdamW
            self.optimizer = optim.AdamW([
                    {'params': decay_params,    'weight_decay': 1e-3},
                    {'params': no_decay_params, 'weight_decay': 0.0,  'lr': 5e-5}
                ], lr=self.learning_rate)
            # self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, mode='max', factor=0.8, patience=500, threshold=1e-3, threshold_mode='abs', min_lr=1e-6, cooldown=300)
            self.scheduler = optim.lr_scheduler.OneCycleLR(self.optimizer, max_lr=[self.learning_rate,5e-5], total_steps=max_iter, pct_start=0.3)
           
   



            self.n_step = 3
            self.n_step_buffer = NStepBuffer(self.n_step, self.gamma)


        #Experience Replay
            if self.replay_type=="PER":
                self.loss=nn.SmoothL1Loss(reduction='none')
                self.memory = PrioritizedExperienceReplay(self.replay_buffer_size,self.device,alpha=0.6,beta=0.4)
                self.beta_increment=(1-0.4) /(max_iter*0.9)
            else:
                self.memory = ExperienceReplay(self.replay_buffer_size,self.device)
                self.loss=nn.MSELoss()

            self.observed_pop=torch.zeros(self.env.number_of_contents,device=self.device)
            


    def create_u_states(self):
        # np.argsort(...)[:, ::-1][:, :M] is all BASIC slicing, so the result is a
        # VIEW that pins the full N x N int64 argsort base (191 MB at N=5000) for
        # the agent's entire lifetime -- including after finalize_training. .copy()
        # materializes just the N x M block that is actually read (42 MB at M=1107,
        # 0.7 MB at M=19). Same values, same dtype, same order.
        u_states = np.argsort(self.env.u, axis=1)[:, ::-1][:, :self.rec_options_num]
        return u_states.copy()


    def find_topN(self, state):
        # Index 0 is the current item (the attention query anchor).
        # Positions 1..K are ALWAYS the K cache items, so DQNFlex's
        # h_cached = h_contextual[:, 1:K+1, :] stays aligned even when the
        # current item is itself cached (it then appears at index 0 AND in 1..K).
        cur   = state[0]
        cache = list(state[1])
        essential = [cur] + cache

        remaining = self.rec_options_num - len(essential)
        if remaining > 0:
            candidates = np.setdiff1d(self.u_states[cur], essential, assume_unique=False)
            essential.extend(candidates[:remaining].tolist())

        # First occurrence wins -> current item maps to 0 even if also cached.
        topN_map = {}
        for idx, item in enumerate(essential):
            if item not in topN_map:
                topN_map[item] = idx
        return essential, topN_map



    def representation_transformation(self,state):
        if self.state_features==5:
            return self.representation_transformation_5(state)
        elif self.state_features==6:
            return self.representation_transformation_6(state)
        elif self.state_features==7:
            return self.representation_transformation_7(state)
        else:
            raise ValueError(f"no state representation for state_features={self.state_features}")


   

    def representation_transformation_5(self, state):
        """
        Optimized state representation addressing environment constraints explicitly:
        1st Feature: Is Current Content Indicator (Binary)
        2nd Feature: Continuous Similarity to Currently Watched Content (Float)
        3rd Feature: Cache Status Mask (Binary)
        4th Feature: Maximum Similarity to the Existing Cache (Float) -> Teaches Cache Diversity
        5th Feature: Global Content Popularity (Float)
        """
        current_content = state[0]
        cache_state = list(state[1])

        # Retrieve the filtered topN items for this state
        topN, topN_map = self.find_topN(state)
        topN_t = torch.tensor(topN, dtype=torch.long, device=self.device)

        

        # 1. Is Current Content Anchor
        is_current = torch.zeros(self.env.number_of_contents, device=self.device)
        is_current[current_content] = 1.0
        feat_is_current = is_current[topN_t].view(-1, 1)

        # 2. Continuous Similarity (Replaces binary threshold mask)
        recommended_tensor = self.recommended_mask[state[0]].float()
        feat_sim = recommended_tensor[topN_t].view(-1, 1)
        

        # 3. Cache Status Mask
        is_cached = torch.zeros(self.env.number_of_contents, device=self.device)
        is_cached[cache_state] = 1.0
        feat_is_cached = is_cached[topN_t].view(-1, 1)


        # 4. Global Popularity
        # feat_pop = self.popularity[topN_t].view(-1, 1).float()
        pop=torch.div(self.observed_pop,torch.clamp(torch.sum(self.observed_pop),min=1.0)).float()
        feat_pop = pop[topN_t].view(-1,1)
        

        
        next_step=self.recommended_mask[:,cache_state].amax(dim=1)[topN_t].view(-1,1).float()


        # Concatenate features into [N_options, 6]
        state_rep = torch.cat((
            feat_is_current,
            feat_sim,
            feat_is_cached,
            feat_pop,
            next_step,
        ), dim=1)

        return state_rep, topN, topN_map




    def representation_transformation_6(self, state):
        current_content = state[0]
        cache_state = list(state[1])

        # Retrieve the filtered topN items for this state
        topN, topN_map = self.find_topN(state)
        topN_t = torch.tensor(topN, dtype=torch.long, device=self.device)
        cache_t = torch.tensor(cache_state, dtype=torch.long, device=self.device)

       

        # 1. Is Current Content Anchor
        feat_is_current = (topN_t == current_content).float().view(-1, 1)

        # 2. Continuous Similarity (Replaces binary threshold mask)
        
        feat_sim = self.recommended_mask[current_content, topN_t].float().view(-1, 1)

        # 3. Cache Status Mask (full-N buffer kept: it also feeds the matvec below)
        is_cached = torch.zeros(self.env.number_of_contents, device=self.device)
        is_cached[cache_state] = 1.0
        feat_is_cached = is_cached[topN_t].view(-1, 1)

        # 4. Global Popularity
        pop=torch.div(self.observed_pop,torch.clamp(torch.sum(self.observed_pop),min=1.0)).float()
        feat_pop = pop[topN_t].view(-1,1)

        # 5. Future cache value (matvec restricted to candidate rows)
        future_cache_value = normalize_tensor(self.recommended_mask[topN_t].float() @ (pop * (1 - is_cached))).view(-1,1).float()

        # 6. Cache utility
        
        next_step = self.recommended_mask[topN_t.unsqueeze(1), cache_t.unsqueeze(0)].amax(dim=1).view(-1,1).float()

        # Concatenate features into [N_options, 6]
        state_rep = torch.cat((
            feat_is_current,
            feat_sim,
            feat_is_cached,
            feat_pop,
            future_cache_value,
            next_step,

        ), dim=1)

        return state_rep, topN, topN_map

    def representation_transformation_7(self, state):
        current_content = state[0]
        cache_state = list(state[1])

        # Retrieve the filtered topN items for this state
        topN, topN_map = self.find_topN(state)
        topN_t = torch.tensor(topN, dtype=torch.long, device=self.device)
        cache_t = torch.tensor(cache_state, dtype=torch.long, device=self.device)


        # 1. Is Current Content Anchor
        feat_is_current = (topN_t == current_content).float().view(-1, 1)

        # 2. Continuous Similarity (Replaces binary threshold mask)
        
        feat_sim_binary = self.recommended_mask[current_content, topN_t].float().view(-1, 1)

        # 3. Cache Status Mask (full-N buffer kept: it also feeds the matvec below)
        is_cached = torch.zeros(self.env.number_of_contents, device=self.device)
        is_cached[cache_state] = 1.0
        feat_is_cached = is_cached[topN_t].view(-1, 1)

        # 4. Global Popularity
        pop=torch.div(self.observed_pop,torch.clamp(torch.sum(self.observed_pop),min=1.0)).float()
        feat_pop = pop[topN_t].view(-1,1)

        # 5. Future cache value (matvec restricted to candidate rows)
        future_cache_value = normalize_tensor(self.recommended_mask[topN_t].float() @ (pop * (1 - is_cached))).view(-1,1).float()

        # 6. Cache utility
        next_step = self.recommended_mask[topN_t.unsqueeze(1), cache_t.unsqueeze(0)].amax(dim=1).view(-1,1).float()

        # 7. Content Similarity Continuous (candidate entries of the u-row only)

        feat_sim_continuous = torch.from_numpy(self.env.u[current_content][topN]).to(device=self.device).float().view(-1, 1)

        # Concatenate features into [N_options, 7]
        state_rep = torch.cat((
            feat_is_current,
            feat_sim_binary,
            feat_is_cached,
            feat_pop,
            future_cache_value,
            next_step,
            feat_sim_continuous

        ), dim=1)

        return state_rep, topN, topN_map


   


    def update_target_network(self):
        """
        Used for soft updating the target network.
        """
        with torch.no_grad():
            for p_target, p_policy in zip(self.q_network_target.parameters(), self.q_network_policy.parameters()):
                p_target.data.copy_((1.0 - self.taf) * p_target.data + self.taf * p_policy.data)
    
    
    def update_target_network_hard(self):
        """
        Used for hard updating the target network.
        """
        with torch.no_grad():
            for p_target, p_policy in zip(self.q_network_target.parameters(), self.q_network_policy.parameters()):
                p_target.data.copy_(p_policy.data)


    
    def save_model(self,path):
        """
        Saves the policy network,tagrget network , the value of epsilon and the experience buffer in a file named model.chkpt .
        """

        torch.save({
                'q_network_policy':self.q_network_policy.state_dict(),
                'q_network_target':self.q_network_target.state_dict(),
                'training_episodes':self.training_episodes,
                'optimizer':self.optimizer.state_dict()


        },os.path.join(path,self.checkpoint))


    def load_model(self,path):
        """
        Loads the policy network,tagrget network , the value of epsilon and the experience buffer from a file named model.chkpt .
        """
        checkpoint=torch.load(os.path.join(path,self.checkpoint),self.device)
        self.q_network_policy.load_state_dict(checkpoint['q_network_policy'])
        self.q_network_target.load_state_dict(checkpoint['q_network_target'])
        self.training_episodes=checkpoint['training_episodes']
        self.optimizer.load_state_dict(checkpoint['optimizer'])

## DQN Agent without caching

In [ ]:
class DQNAgent_NC(DQNAgent):

    def __init__(self,env,state_features=3,learning_rate=0.1,gamma=0.9,eps=0.9,max_iter=50000,replay_type="PER",replay_buffer_size=10000,batch_size=32,taf=0.01,rec_only=True):
        super().__init__(env,state_features,learning_rate,gamma,eps,max_iter,replay_type,replay_buffer_size,batch_size,taf,rec_only)
        self.number_of_actions=self.rec_options_num
        self.eps_decay_rate=1e-4




    def choose_actions(self, state,state_rep,topN,topN_map):
        """
        RL recommendations plus popularity based caching.

        """

        action = []
        # imitation_prob=self.imitation_prob(self.training_episodes)
        if self.q_network_policy.training:
            self.q_network_policy.reset_noise()

       

        with torch.no_grad():

            q_network_output=self.q_network_policy(state_rep)
            current_state=topN_map[state[0]]

            mask_actions = torch.zeros(1,self.number_of_actions, dtype=torch.bool,device=self.device)

            for _i, _item in enumerate(topN):
                if _item == state[0]:
                    mask_actions[0][_i] = True



            q_masked = torch.where(mask_actions, torch.tensor([-1e8], device=self.device), q_network_output.to(self.device))
            chosen_action = int(torch.argmax(q_masked).item())

            cache=state[1]
            next_state_rec=topN[chosen_action]

            if state[0] not in state[1]:
                pop=torch.div(self.observed_pop,torch.clamp(torch.sum(self.observed_pop),min=1.0)).float()
                if pop[state[0]]>torch.min(pop[cache]):
                    index=torch.argmin(pop[cache])
                    cache=generate_cache_state(self.env.cache_size,state,index)

            next_state=next_state_rec,cache
            action.append(next_state)


        return action,torch.max(q_masked)


    def learn(self, experiences):

            self.q_network_policy.reset_noise()


            if self.memory.__class__ == PrioritizedExperienceReplay:
                _ , actions, rewards, dones, next_states, states_batch, next_states_batch, idx, weights ,_, topN_map,_,next_topN_map = experiences
            else:
                _ , actions, rewards, dones, next_states, states_batch, next_states_batch, _, topN_map, _,next_topN_map = experiences


            # Start autocast for mixed precision during forward and loss computation
            with  torch.autocast(device_type=self.device.type):
                policy_action = self.q_network_policy(states_batch)

                action_contents = actions[0].tolist()

                action_indices = []
                for top_map,a in zip(topN_map,action_contents):
                    action_indices.append(top_map[a])

                predicted_action = policy_action[torch.arange(self.batch_size,device=self.device), torch.tensor(action_indices, device=self.device)]



                with torch.no_grad():
                    _was_training = self.q_network_policy.training
                    self.q_network_policy.eval()
                    next_actions = self.q_network_policy(next_states_batch)
                    if _was_training:
                        self.q_network_policy.train()
                    mask_actions = torch.zeros_like(next_actions, dtype=torch.bool, device=self.device)

                    next_contents = next_states[0].tolist()   # one transfer


                    next_caches = next_states[1].tolist()
                    
                    rows = torch.arange(self.batch_size, device=self.device)
                    cur_idx = torch.tensor([next_topN_map[i][next_contents[i]] for i in range(self.batch_size)], device=self.device)
                    cache_pos = torch.tensor([1 + next_caches[i].index(next_contents[i]) if next_contents[i] in next_caches[i] else -1 for i in range(self.batch_size)],device=self.device)
                    mask_actions[rows, cur_idx] = True
                    _has = cache_pos >= 0
                    if _has.any():
                        mask_actions[rows[_has], cache_pos[_has]] = True

                    q_masked = torch.where(~mask_actions, next_actions, torch.tensor(torch.finfo(next_actions.dtype).min, device=self.device))
                    best_actions = q_masked.argmax(1, keepdim=True)

                    target_q_action=self.q_network_target(next_states_batch)

                    target_q=target_q_action.gather(1,best_actions)



                # target_action = rewards.view(self.batch_size,-1)+ self.gamma * target_q.view(self.batch_size, -1)
                    target_action = (rewards.view(self.batch_size, -1)
                     + (self.gamma ** self.n_step)           # <-- key change
                     * (1 - dones.float()).view(self.batch_size, -1)
                     * target_q.view(self.batch_size, -1))



                loss = self.loss(predicted_action.view(-1), target_action.view(-1))




            if self.memory.__class__ == PrioritizedExperienceReplay:
                td_errors = (target_action.view(-1) - predicted_action.view(-1)).detach()
                new_priorities = (torch.abs(torch.nan_to_num(td_errors, nan=0.0, posinf=1e3, neginf=-1e3)) + self.memory.eps).clamp(max=1e3).tolist()
                # new_priorities = (torch.abs(td_errors) + self.memory.eps).tolist()
                self.memory.update_priority(idx, new_priorities)
                self.memory.beta_increase(self.beta_increment)
                loss = torch.mean(weights.squeeze().float()* loss.float())



            self.optimizer.zero_grad()
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.q_network_policy.parameters(), max_norm=5.0)
            self.scaler.step(self.optimizer)
            self.scaler.update()


            return loss.item()

In [ ]:
class DQNAgent_NC_PBRS(DQNAgent_NC):
    """
    DQNAgent_NC + potential-based reward shaping (PBRS), same Phi/scale as
    DQNAgent_WC.learn. NC still doesn't control cache admission (popularity
    heuristic in choose_actions, inherited unchanged) -- but Phi is a function
    of state only, so Ng/Harada/Russell (1999) policy-invariance holds
    regardless of what drives the cache. Isolates the reward-density effect
    from the joint-action-space effect in the WC-vs-NC comparison.

    reward_version: 'nc_pbrs_v1' -- a new experiment family. NOT comparable to
    plain DQNAgent_NC runs (different reward function); cache-hit MICRO stays
    comparable since it's read from env hit-tracking, not the (shaped) reward.
    """

    def learn(self, experiences):

            self.q_network_policy.reset_noise()


            if self.memory.__class__ == PrioritizedExperienceReplay:
                states , actions, rewards, dones, next_states, states_batch, next_states_batch, idx, weights ,_, topN_map,_,next_topN_map = experiences
            else:
                states , actions, rewards, dones, next_states, states_batch, next_states_batch, _, topN_map, _,next_topN_map = experiences

            # Potential-based reward shaping -- identical Phi/scale to
            # DQNAgent_WC.learn; see class docstring.
            current_items = states[0].tolist()
            next_items = next_states[0].tolist()
            states_cache = states[1].tolist()
            next_caches = next_states[1].tolist()

            cur_arr    = np.asarray(current_items)
            nxt_arr    = np.asarray(next_items)
            cache_arr  = np.asarray(states_cache)
            ncache_arr = np.asarray(next_caches)

            pop=torch.div(self.observed_pop,torch.clamp(torch.sum(self.observed_pop),min=1.0)).float()
            secured_cur  = self.recommended_mask[cur_arr[:, None], cache_arr].float().amax(dim=1)   # (B,)
            secured_nxt  = self.recommended_mask[nxt_arr[:, None], ncache_arr].float().amax(dim=1)  # (B,)
            # fallback channel: popularity coverage of the cache
            fallback_cur = pop[cache_arr].sum(dim=1)                                              # (B,)
            fallback_nxt = pop[ncache_arr].sum(dim=1)                                             # (B,)

            current_potential = secured_cur + (1.0 - secured_cur) * fallback_cur                # (B,)
            next_potential    = secured_nxt + (1.0 - secured_nxt) * fallback_nxt

            not_done = (1.0 - dones.float().view(-1))
            shaping_reward = 0.1 * (self.gamma ** self.n_step * next_potential * not_done - current_potential)
            augmented_rewards = rewards + shaping_reward


            # Start autocast for mixed precision during forward and loss computation
            with  torch.autocast(device_type=self.device.type):
                policy_action = self.q_network_policy(states_batch)

                action_contents = actions[0].tolist()

                action_indices = []
                for top_map,a in zip(topN_map,action_contents):
                    action_indices.append(top_map[a])

                predicted_action = policy_action[torch.arange(self.batch_size,device=self.device), torch.tensor(action_indices, device=self.device)]



                with torch.no_grad():
                    _was_training = self.q_network_policy.training
                    self.q_network_policy.eval()
                    next_actions = self.q_network_policy(next_states_batch)
                    if _was_training:
                        self.q_network_policy.train()
                    mask_actions = torch.zeros_like(next_actions, dtype=torch.bool, device=self.device)

                    next_contents = next_states[0].tolist()   # one transfer


                    next_caches = next_states[1].tolist()
                    # [PERF row B] vectorised next-state mask (was a 32-iter Python loop of
                    #   tiny per-sample GPU scatters). Bitwise-identical bool mask.
                    rows = torch.arange(self.batch_size, device=self.device)
                    cur_idx = torch.tensor(
                        [next_topN_map[i][next_contents[i]] for i in range(self.batch_size)],
                        device=self.device)
                    cache_pos = torch.tensor(
                        [1 + next_caches[i].index(next_contents[i]) if next_contents[i] in next_caches[i] else -1
                         for i in range(self.batch_size)],
                        device=self.device)
                    mask_actions[rows, cur_idx] = True
                    _has = cache_pos >= 0
                    if _has.any():
                        mask_actions[rows[_has], cache_pos[_has]] = True

                    q_masked = torch.where(~mask_actions, next_actions, torch.tensor(torch.finfo(next_actions.dtype).min, device=self.device))
                    best_actions = q_masked.argmax(1, keepdim=True)

                    target_q_action=self.q_network_target(next_states_batch)

                    target_q=target_q_action.gather(1,best_actions)



                    target_action = (augmented_rewards.view(self.batch_size, -1)
                     + (self.gamma ** self.n_step)
                     * (1 - dones.float()).view(self.batch_size, -1)
                     * target_q.view(self.batch_size, -1))



                loss = self.loss(predicted_action.view(-1), target_action.view(-1))




            if self.memory.__class__ == PrioritizedExperienceReplay:
                td_errors = (target_action.view(-1) - predicted_action.view(-1)).detach()
                new_priorities = (torch.abs(torch.nan_to_num(td_errors, nan=0.0, posinf=1e3, neginf=-1e3)) + self.memory.eps).clamp(max=1e3).tolist()
                self.memory.update_priority(idx, new_priorities)
                self.memory.beta_increase(self.beta_increment)
                loss = torch.mean(weights.squeeze().float()* loss.float())



            self.optimizer.zero_grad()
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.q_network_policy.parameters(), max_norm=5.0)
            self.scaler.step(self.optimizer)
            self.scaler.update()


            return loss.item()

## DQN Agent with caching

In [ ]:
class DQNAgent_WC(DQNAgent):

    def __init__(self,env,state_features=3,learning_rate=0.1,gamma=0.9,eps=0.9,max_iter=50000,replay_type="PER",replay_buffer_size=10000,batch_size=32,taf=0.01,rec_only=False):
        super().__init__(env,state_features,learning_rate,gamma,eps,max_iter,replay_type,replay_buffer_size,batch_size,taf,rec_only)
        self.number_of_actions=self.rec_options_num*(self.env.cache_size+1)
        self.cache_op=np.zeros(self.env.cache_size+1,dtype=np.int64)
        
        


    def choose_actions(self, state,state_rep,topN,topN_map):
        action = []
        if self.q_network_policy.training:
            self.q_network_policy.reset_noise()

  
        with torch.no_grad():

            q_network_output=self.q_network_policy(state_rep)

            mask_actions = torch.zeros(1,self.number_of_actions, dtype=torch.bool,device=self.device)

            for _i, _item in enumerate(topN):
                if _item == state[0]:
                    _blk = _i * (self.env.cache_size + 1)
                    mask_actions[0][_blk:_blk + self.env.cache_size + 1] = True
            if state[0] in state[1]:
                for k in range(self.env.cache_size):
                    mask_actions[0][range(k,self.number_of_actions,(self.env.cache_size+1))]= True
                

            q_masked = torch.where(mask_actions, torch.tensor([-1e8], device=self.device), q_network_output.to(self.device))
            chosen_action = torch.argmax(q_masked).item()
            
            next_state=action_format(self.env.cache_size,state,chosen_action,topN)
            self.cache_op[chosen_action % (self.env.cache_size+1)]+=1

            action.append(next_state)




        return action,torch.max(q_masked)


    def learn(self, experiences):

            self.q_network_policy.reset_noise()
            if self.memory.__class__ == PrioritizedExperienceReplay:
                states, actions, rewards, dones, next_states, states_batch, next_states_batch, idx, weights, _ , topN_map, _,topN_map_next = experiences
            else:
                states , actions, rewards, dones, next_states, states_batch, next_states_batch, _, topN_map, _, topN_map_next = experiences


            
            current_items = states[0].tolist()
            next_items = next_states[0].tolist()
            states_cache = states[1].tolist()
            next_caches = next_states[1].tolist()
            
            

            # Potential-based reward shaping (policy-invariant).
            # Phi(s) = sum_{c in cache} u[current_item, c] * popularity[c]; cache is size K.
            cur_arr    = np.asarray(current_items)
            nxt_arr    = np.asarray(next_items)
            cache_arr  = np.asarray(states_cache)
            ncache_arr = np.asarray(next_caches)

        

            pop=torch.div(self.observed_pop,torch.clamp(torch.sum(self.observed_pop),min=1.0)).float()
            secured_cur  = self.recommended_mask[cur_arr[:, None], cache_arr].float().amax(dim=1)   # (B,)
            secured_nxt  = self.recommended_mask[nxt_arr[:, None], ncache_arr].float().amax(dim=1)  # (B,)
            # fallback channel: popularity coverage of the cache
            fallback_cur = pop[cache_arr].sum(dim=1)                                              # (B,)
            fallback_nxt = pop[ncache_arr].sum(dim=1)                                             # (B,)

            current_potential = secured_cur + (1.0 - secured_cur) * fallback_cur                # (B,)
            next_potential    = secured_nxt + (1.0 - secured_nxt) * fallback_nxt  


            not_done = (1.0 - dones.float().view(-1))
            shaping_reward = 0.1 * (self.gamma ** self.n_step * next_potential * not_done - current_potential)
            # print(torch.max(torch.abs(shaping_reward)))
            augmented_rewards = rewards + shaping_reward

            # print(augmented_rewards)



          # Start autocast for mixed precision during forward and loss computation
            with  torch.autocast(device_type=self.device.type):
                _was_training = self.q_network_policy.training
                self.q_network_policy.eval()
                next_actions = self.q_network_policy(next_states_batch)
                if _was_training:
                    self.q_network_policy.train()
                policy_action = self.q_network_policy(states_batch)

                action_contents = actions[0].tolist()
                action_caches   = actions[1].tolist()    # next_cache list
               

                action_indices = []
                for top_map, a_content, a_cache, s_cache in zip(topN_map, action_contents, action_caches, states_cache):
                    # Reconstruct which cache slot was replaced
                    slot = self.env.cache_size  # default: no change
                    for index, item in enumerate(s_cache):
                        if item not in a_cache:
                            slot = index
                            break
                    content_idx = top_map[a_content]
                    action_indices.append(content_idx * (self.env.cache_size + 1) + slot)

                predicted_action = policy_action[torch.arange(self.batch_size,device=self.device), torch.tensor(action_indices, device=self.device)]



                with torch.no_grad():
                    next_actions = self.q_network_policy(next_states_batch)
                    mask_actions = torch.zeros_like(next_actions, dtype=torch.bool, device=self.device)


                    next_contents = next_states[0].tolist()
                    next_caches = next_states[1].tolist()

                    _K1 = self.env.cache_size + 1
                    _M  = next_actions.shape[1] // _K1
                    _m3 = mask_actions.view(self.batch_size, _M, _K1)
                    rows = torch.arange(self.batch_size, device=self.device)
                    nsi = torch.tensor([topN_map_next[i][next_contents[i]] for i in range(self.batch_size)], device=self.device)
                    p   = torch.tensor([1 + next_caches[i].index(next_contents[i]) if next_contents[i] in next_caches[i] else -1 for i in range(self.batch_size)], device=self.device)
                    _m3[rows, nsi, :] = True
                    _cached = p >= 0
                    if _cached.any():
                        _m3[rows[_cached], p[_cached], :] = True
                        _m3[_cached, :, :self.env.cache_size] = True

                   
                    q_masked = torch.where(mask_actions, torch.tensor(torch.finfo(next_actions.dtype).min, device=self.device),  next_actions)
                   
                    best_actions = q_masked.argmax(1, keepdim=True)
                    
                    target_q_action=self.q_network_target(next_states_batch)

                    target_q=target_q_action.gather(1,best_actions)


                    target_action = (augmented_rewards.view(self.batch_size, -1)
                     + (self.gamma ** self.n_step)           # <-- key change
                     * (1 - dones.float()).view(self.batch_size, -1)
                     * target_q.view(self.batch_size, -1))

                loss = self.loss(predicted_action.view(-1), target_action.view(-1))


            if self.memory.__class__ == PrioritizedExperienceReplay:
                td_errors = (target_action.view(-1) - predicted_action.view(-1)).detach()
                new_priorities = (torch.abs(td_errors) + self.memory.eps).tolist()
                self.memory.update_priority(idx, new_priorities)
                self.memory.beta_increase(self.beta_increment)
                loss = torch.mean(weights.squeeze().float()* loss.float())

            else:
                loss=loss.mean()



            self.optimizer.zero_grad()
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.q_network_policy.parameters(), max_norm=5.0)
            self.scaler.step(self.optimizer)
            self.scaler.update()


            return loss.item()

# DQN Train Function

In [ ]:
def get_current_lr(optimizer):

    for param_group in optimizer.param_groups:

        return param_group['lr']

def train(env,agent,plot_handle,max_iter = 1000,plot_interval=2000,threshold = 0.03,eps_decay=False,path=None,train_enable=True,save_load=False,ckpt_window=1000):

    costs = []
    episodes=0
    
    start_time = time.time()

    list_of_rewards = []
    list_of_recommendation_rewards=[]
    list_of_caching_rewards=[]
    list_of_cache_hit=[]
    list_of_max_q=[]
    list_of_sigma=[]
    total_hits=0
    total_steps=0
    best_macro=-1.0
    best_ep=-1
    if(path is not None and os.path.isfile(path+agent.checkpoint) and save_load==True):
        agent.load_model(path)
        print(agent.training_episodes)
        print("Checkpoint loaded!")

    agent.q_network_policy.train()




    if( not train_enable):
           agent.q_network_policy.eval()


    state=None
    for episodes in trange(1, max_iter):

        if state:
            state = env.refresh()[0],state[1]
        else:
            state = env.refresh()
        i = 0

        done = False

        current_reward = 0
        current_loss=[]
        current_q=[]
        current_recommendation_reward=0
        current_caching_reward=0
        cache_hit=0




        state_rep=None
        next_state_rep=None


        agent.n_step_buffer.clear()
        while (not done):


            if next_state_rep is None:
              state_rep,topN,topN_map=agent.representation_transformation(state)
            else:
              # Share the previous step's objects rather than copying them. Nothing
              # mutates state_rep / topN / topN_map in place after construction:
              # representation_transformation builds fresh objects on every call and
              # every downstream use (learn, choose_actions, n_step_buffer) is a read.
              # The copies only doubled what the replay buffer retains -- measured
              # 1.88 GB -> 0.94 GB host and 591 -> 296 MB GPU at M=1107. The values
              # learn() sees are unchanged.
              state_rep=next_state_rep
              topN=next_topN
              topN_map=next_topN_map



            action,max_q=agent.choose_actions(state,state_rep,topN,topN_map)
            current_q.append(max_q.item())


            next_state,reward,done=env.simulate(action,state)

            agent.observed_pop[next_state[0]]+=1

            if(train_enable==True):

                next_state_rep,next_topN,next_topN_map=agent.representation_transformation(next_state)
                exp=Experience(state,action,reward,done,next_state,state_rep,next_state_rep,topN,topN_map,next_topN,next_topN_map)
                
                agent.n_step_buffer.append(exp)
                if agent.n_step_buffer.is_ready():
                    n_step_exp = agent.n_step_buffer.get()
                    agent.memory.append(n_step_exp)


                if(agent.memory.__len__()>=agent.batch_size*100):
                        experiences=agent.memory.sample(agent.batch_size)
                        current_loss.append(agent.learn(experiences))
                



            if action[0][0]==next_state[0]:
                current_recommendation_reward+=1
            else:
                current_recommendation_reward+=0

            if action[0][0] in state[1]:
              current_caching_reward+=1
            else:
              current_caching_reward+=0

            if next_state[0] in state[1]:
                cache_hit+=1

            state=next_state

            i+=1
            current_reward+=reward




        if(i!=0):
            # Hits are counted from the env's hit-tracking (cache_hit), never from the
            # reward (experiment-manager S4.4) -- equal today (0/1 hit reward), but a
            # reward change must not silently change the metric.
            total_hits+=cache_hit
            total_steps+=i
            costs.append(sum(current_loss)/len(current_loss) if current_loss else 0.0)
            # PER-EPISODE hit rate (this episode's hits / its steps), as in done_seedsweep.ipynb
            # and multi_cache_setup.train_dqn. This list feeds the checkpoint rule below, the
            # MACRO number and the returned train curve. The cumulative total_hits/total_steps
            # that had drifted in here after the 2026-07-15 sweep is reported once at the end
            # as MICRO instead.
            list_of_rewards.append(cache_hit/i)
            list_of_cache_hit.append(current_reward/i)
            list_of_max_q.append(np.mean(current_q))
            list_of_sigma.append(float(np.mean([m.weight_sigma.detach().abs().mean().item() for m in agent.q_network_policy.modules() if hasattr(m, 'weight_sigma')] or [0.0])))
            list_of_recommendation_rewards.append(100*(current_recommendation_reward/i))
            list_of_caching_rewards.append(100*(current_caching_reward/i))
            
        


        if train_enable:
            agent.training_episodes+=1
            # agent.scheduler.step(np.mean(list_of_rewards[-100:]))
            if current_loss :
                agent.scheduler.step()
            # agent.update_target_network()
            if (episodes+1) % 500==0  and costs:
                agent.update_target_network_hard()
            # Checkpoint selection = the multi-cache rule (multi_cache_setup.train_dqn, 2026-07-25),
            # adopted here 2026-09-17: best checkpoint = highest `ckpt_window`-episode rolling mean
            # of the PER-EPISODE env hit rate, and no checkpoint before half of training. A rolling
            # mean of the cumulative ratio spikes in the first few hundred episodes and pins
            # best_state_dict to a barely-trained net (multi-cache seed 11: ep209/40000).
            # checkpoint_selection_metric = 'per_episode_roll{ckpt_window}_halfgate'. The 2026-07-15
            # seed sweep (done_seedsweep.ipynb) used roll200 / no gate; project decision 2026-09-17:
            # the rule change is treated as having NO effect on comparability -- do not split
            # single-edge runs into families by it.
            if len(list_of_rewards) >= ckpt_window:
                _roll=float(np.mean(list_of_rewards[-ckpt_window:]))
                if _roll > best_macro and episodes > (max_iter / 2):
                    best_macro=_roll; best_ep=episodes
                    agent.best_state_dict={k:v.detach().cpu().clone() for k,v in agent.q_network_policy.state_dict().items()}

       

        if((episodes+1) % plot_interval == 0):

            converged =_periodic_plot(episodes,plot_interval,list_of_rewards,costs=costs,
                           extra_lists=[list_of_cache_hit,list_of_recommendation_rewards,list_of_caching_rewards,list_of_max_q,list_of_sigma],
                           extra_titles=["Cache hit MACRO (per episode)","Recommendation Acceptance Rate","Recommending Cached Content Rate","Mean of Max Q per Episode","Mean NoisyNet sigma"],
                           episode_slices=500,threshold=threshold,early_stopage=train_enable,plot_handle=plot_handle)

            # print("eps= ",agent.eps)
            if train_enable:
                print("LR= " ,get_current_lr(agent.optimizer))
                


            if(save_load==True and train_enable==True):
                agent.save_model(path)

            if converged:
                break




    running_time=time.time()-start_time
    if train_enable==True:
        finalize_training(agent)

    macro=float(np.mean(list_of_rewards)) if list_of_rewards else 0.0
    micro=(total_hits/total_steps) if total_steps else 0.0
    agent.last_eval_macro=macro
    agent.last_eval_micro=micro
    print(f"[{'TRAIN' if train_enable else 'EVAL'}] cache-hit  MACRO(per-episode mean)={macro:.4f}  MICRO(total hits/steps = standard cache-hit ratio)={micro:.4f}")
    if train_enable and best_ep>=0:
        print(f"[TRAIN] best-checkpoint {ckpt_window}-ep rolling per-episode hit rate={best_macro:.4f} @ep{best_ep}  (stored in agent.best_state_dict; load it before eval to deploy the best policy)")

    return running_time,episodes,costs,list_of_rewards

# Non RL Impementation

### Baseline

In [ ]:
class Non_RL_agent_baseline:

     def __init__(self,env):
        self.env=env
        self.observed_pop=torch.zeros(self.env.number_of_contents)
        


     def choose_actions(self,state):
         action=[]
         cache=state[1]
         i=0
         if state[0] not in state[1]:
                  pop=torch.div(self.observed_pop,torch.clamp(torch.sum(self.observed_pop),min=1.0)).float()
                  if pop[state[0]]>torch.min(pop[cache]):
                     index=torch.argmin(pop[cache])
                     cache=generate_cache_state(self.env.cache_size,state,index)

         next_item_choice=np.argmax(self.env.u[state[0]])
         next_state=next_item_choice,cache

         assert check_action_DQN(state,next_state), "Illegal action"


         action.append(next_state)

         return action

### Greedy

In [ ]:
class Non_RL_agent_greedy:

     def __init__(self,env):
        self.env=env
        self.observed_pop=torch.zeros(self.env.number_of_contents)

     def choose_actions(self,state):
        action=[]
        cache=state[1]
        i=0
        if state[0] not in state[1]:
                pop=torch.div(self.observed_pop,torch.clamp(torch.sum(self.observed_pop),min=1.0)).float()
                if pop[state[0]]>torch.min(pop[cache]):
                    index=torch.argmin(pop[cache])
                    cache=generate_cache_state(self.env.cache_size,state,index)
                    
        next_item_choices = np.argsort(self.env.u[state[0]][list(state[1])])[::-1]

        next_item=state[1][next_item_choices[i]]
        next_state=next_item,cache

        while check_action_DQN(state,next_state)==False :
            i+=1
            next_item=state[1][next_item_choices[i]]
            next_state=next_item,cache



        action.append(next_state)

        return action

## Udpate

In [ ]:
def train_non_RL(env,agent,plot_handle,max_iter = 1000,plot_interval=2000,threshold = 1e-3,eps_decay=False,path=None,train_enable=True,save_load=False):


    episodes=0

    start_time = time.time()
    total_steps=0
    total_hits=0

    list_of_rewards = []
    list_of_recommendation_rewards=[]
    list_of_caching_rewards=[]


    state=None
    for episodes in trange(1, max_iter):

        if state:
            state = env.refresh()[0],state[1]
        else:
            state = env.refresh()
        i = 0
        cache_hit=0

        done = False

        current_penalty = 0
        current_reward = 0
        current_loss=0
        current_recommendation_reward=0
        current_caching_reward=0

        while (not done):

            action=agent.choose_actions(state)

            next_state,reward,done=env.simulate(action,state)

            agent.observed_pop[next_state[0]]+=1


            i+=1
            current_reward+=reward


            if reward == env.rewards[0]:
                current_penalty+=1

            if action[0][0]==next_state[0]:
                current_recommendation_reward+=1
            else:
                current_recommendation_reward+=0

            if action[0][0] in state[1]:
              current_caching_reward+=1
            else:
              current_caching_reward+=0

            if next_state[0] in state[1]:
                cache_hit+=1

            state=next_state




        if(i!=0):
            total_hits+=current_reward
            total_steps+=i
            list_of_rewards.append(total_hits/total_steps)
            list_of_recommendation_rewards.append(current_recommendation_reward/i)
            list_of_caching_rewards.append(current_caching_reward/i)



        if((episodes+1) % plot_interval == 0):

            _periodic_plot(episodes,plot_interval,list_of_rewards,costs=None,
                           extra_lists=[list_of_recommendation_rewards,list_of_caching_rewards],
                           extra_titles=["Recommendation Acceptance Rate","Recommending Cached Content Rate"],
                           episode_slices=500,threshold=threshold,early_stopage=False,plot_handle=plot_handle)



    running_time=time.time()-start_time
    return running_time,episodes,list_of_rewards

# Testing

In [ ]:
list_of_results=[]
pre_trained_results=[]

# --- Dataset selection --------------------------------------------------------
# 'movielens' -> ratings.csv              (userId, movieId, rating)
# 'kuairec'   -> kuairec_small_matrix.csv (user_id, video_id, watch_ratio; auto-normalized
#                by normalize_ratings_schema -- see cell defining get_top_movies/create_u)
# 'synthetic' -> no file; create_u/create_popularity fall back to their built-in
#                random-uniform / Zipf generators (dataset=None branch)
DATASET_KIND = 'movielens'

_DATASET_DEFAULTS = {
    'movielens': dict(file='ratings.csv',              N=5000, corr_threshold=0.6),
    'kuairec':   dict(file='kuairec_small_matrix.csv',  N=3000, corr_threshold=0.20),  # full catalog is 3327
    'synthetic': dict(file=None,                        N=5000, corr_threshold=0.8),
}
_dcfg = _DATASET_DEFAULTS[DATASET_KIND]
dataset = _dcfg['file']

u_file = f'u_file_{DATASET_KIND}.npy'
popularity_file = f'popularity_file_{DATASET_KIND}.npy'
path=""

set_seed(7)


In [ ]:
number_of_contents=_dcfg['N']
cache_size=10
rewards=[0,1]
number_of_recommendations=1
max_iter=10000
testing_max_iter=10000
eps_value=0
eps_decay=False
corr_threshold=_dcfg['corr_threshold']
gamma=0.95
converge_threshold=0
prob_to_leave=0.05



# --- Load u/popularity from their cached .npy files if present and sized correctly
#     for this number_of_contents; otherwise (re)build from the selected dataset (or
#     the synthetic fallback if dataset is None) and save. u_file/popularity_file
#     already encode DATASET_KIND (previous cell), so a cached file only ever matches
#     a run of the SAME dataset. The ratings file is read/normalized at most ONCE and
#     the SAME top-content ordering is reused for whichever of u/popularity actually
#     needs rebuilding, so index i means the same content in both.


_ml_cache = {}
def _shared_ratings():
    if 'df' not in _ml_cache:
        # Read ONLY the columns the CF pipeline consumes. The full read also pulled
        # `timestamp` -- a large int64 column that get_top_movies and create_u drop
        # immediately anyway -- which on the 676 MB MovieLens ratings.csv is a big
        # share of the frame. NOTE: no dtype coercion here on purpose. Values are
        # parsed exactly as before (float64), so u / popularity stay bit-identical;
        # only columns that were going to be discarded are never materialized.
        try:
            _df = pd.read_csv(dataset, usecols=['userId', 'movieId', 'rating'])
        except ValueError:
            try:
                _df = pd.read_csv(dataset, usecols=['user_id', 'video_id', 'watch_ratio'])
            except ValueError:
                _df = pd.read_csv(dataset)
        _df = normalize_ratings_schema(_df)
        _ml_cache['df']  = _df
        _ml_cache['top'] = get_top_movies(number_of_contents, df=_df)
    return _ml_cache['df'], _ml_cache['top']

def _build_u():
    if dataset is not None:
        _df, _top = _shared_ratings()
        return create_u(number_of_contents, dataset, top_movies=_top, df=_df)
    return create_u(number_of_contents)

def _build_popularity():
    if dataset is not None:
        _df, _top = _shared_ratings()
        return create_popularity(number_of_contents, dataset, top_movies=_top, df=_df)
    return create_popularity(number_of_contents)

if os.path.isfile(u_file):
    u = np.load(u_file)
    if u.shape[1] != number_of_contents:
        u = _build_u()
        print("U file rebuilt (number_of_contents changed).")
        np.save(u_file, u)
    else:
        print("U file loaded!")
else:
    u = _build_u()
    print("U file created!")
    np.save(u_file, u)

if os.path.isfile(popularity_file):
    popularity = np.load(popularity_file)
    if popularity.shape[0] != number_of_contents:
        popularity = _build_popularity()
        print("Popularity file rebuilt (number_of_contents changed).")
        np.save(popularity_file, popularity)
    else:
        print("Popularity file loaded!")
else:
    popularity = _build_popularity()
    print("Popularity file created!")
    np.save(popularity_file, popularity)





# The ratings frame is only needed while (re)building u / popularity. Once both
# exist it is dead weight (1 GB+ for MovieLens) that would otherwise be held for the
# whole session, so drop it before the env and the agents allocate. No-op on the
# normal path where both .npy caches loaded and nothing was rebuilt.
_ml_cache.clear()
import gc as _gc; _gc.collect()

env=Environment(number_of_contents,cache_size,rewards,number_of_recommendations,corr_threshold,1-gamma,user_type='quality_aware',popularity=popularity,u=u)


#same performance // threshold = 0.5 // scenario 2 // high correlation for cached states
# u[0]=[0,0.3,0.4,0.2]
# u[1]=[0.7,0,0.8,0.4]
# u[2]=[0.6,0.8,0,0.7]
# u[3]=[0.2,0.3,0.4,0]

# caching agent better perfomance // threshold = 0.5 // scenario 1 // low correlation for cached states
# u[0]=[0,0.7,0.4,0.8]
# u[1]=[0.25,0,0.3,0.4]
# u[2]=[0.25,0.25,0,0.4]
# u[3]=[0.6,0.8,0.7,0]


# u[0]=[0,0.3,0.6,0.2]
# u[1]=[0.7,0,0.4,0.1]
# u[2]=[0.4,0.2,0,0.8]
# u[3]=[0.3,0.9,0.4,0]

# u[0]=[0,0.15,0.65,0.07]
# u[1]=[0.53,0,0.05,0.50]
# u[2]=[0.03,0.43,0,0.09]
# u[3]=[0.42,0.82,0.12,0]


print(u.mean())

# print(u)
# print(popularity)

## Non RL Testing

### Baseline

In [ ]:


Agent = Non_RL_agent_baseline(env)
plot_baseline = display(display_id=True)
running_time,episodes,list_of_rewards = train_non_RL(env,Agent,plot_baseline,max_iter=testing_max_iter)
costs=[]
res=Result(list_of_rewards,costs,episodes,running_time)
pre_trained_results.append(res)

### Greedy

In [ ]:

Agent = Non_RL_agent_greedy(env)
plot_greedy = display(display_id=True)
running_time,episodes,list_of_rewards = train_non_RL(env,Agent,plot_greedy,max_iter=testing_max_iter)
costs=[]
res=Result(list_of_rewards,costs,episodes,running_time)
pre_trained_results.append(res)

## Policy Iteration Testing

In [ ]:
# PI_env=Environment(number_of_contents,cache_size,rewards,number_of_recommendations,corr_threshold,1-gamma,user_type='quality_aware',u=u,popularity=popularity,tabular=True)
# PI_agent=Agent_PI(PI_env,number_of_contents,cache_size,corr_threshold,u,gamma=gamma,max_iter=10000,threshold=1e-6,popularity=popularity)
# running_time,episodes,list_of_rewards=update_PI(PI_env,PI_agent,max_iter=testing_max_iter)
# costs=[]
# res=Result(list_of_rewards,costs,episodes,running_time)
# pre_trained_results.append(res)

## TABULAR TESTING

### Tabular without caching

In [ ]:
# env_tab_1=Environment(number_of_contents,cache_size,rewards,number_of_recommendations,corr_threshold,1-gamma,user_type='quality_aware',u=u,tabular=True)
# RL_1=Agent_without_caching(env_tab_1,learning_rate=0.01,gamma=gamma,max_iter=max_iter,eps=eps_value)
# running_time,episodes,costs,list_of_rewards=update(env_tab_1,RL_1,max_iter=max_iter,eps_decay=eps_decay,threshold=converge_threshold,train_enable=True)
# res=Result(list_of_rewards,costs,episodes,running_time)
# list_of_results.append(res)

In [ ]:
# running_time,episodes,costs,list_of_rewards=update(env_tab_1,RL_1,max_iter=testing_max_iter,eps_decay=eps_decay,threshold=converge_threshold,train_enable=False)
# res=Result(list_of_rewards,costs,episodes,running_time)
# pre_trained_results.append(res)

### Tabular with caching

In [ ]:
# env_tab_2=Environment(number_of_contents,cache_size,rewards,number_of_recommendations,corr_threshold,1-gamma,user_type='quality_aware',u=u,tabular=True)
# RL_2=Agent_with_caching(env_tab_2,learning_rate=0.01,gamma=gamma,max_iter=max_iter,eps=eps_value)
# running_time,episodes,costs,list_of_rewards=update(env_tab_2,RL_2,max_iter=max_iter,eps_decay=eps_decay,threshold=converge_threshold,train_enable=True)
# res=Result(list_of_rewards,costs,episodes,running_time)
# list_of_results.append(res)

In [ ]:
# running_time,episodes,costs,list_of_rewards=update(env_tab_2,RL_2,max_iter=testing_max_iter,eps_decay=eps_decay,threshold=converge_threshold,train_enable=False)
# res=Result(list_of_rewards,costs,episodes,running_time)
# pre_trained_results.append(res)

## DQN TEST

### DQN without caching

In [ ]:
batch_size=32
max_iter=20000


RL_DQN_NC=DQNAgent_NC(env,state_features=7,learning_rate=1e-4,gamma=gamma,eps=eps_value,max_iter=max_iter,replay_type="PER",replay_buffer_size=10_000,taf=0.005)

plot_dqnnc_train = display(display_id=True)
running_time,episodes,costs,list_of_rewards=train(env,RL_DQN_NC,max_iter=max_iter,eps_decay=eps_decay,threshold=converge_threshold,path=path,train_enable=True,save_load=False,plot_handle=plot_dqnnc_train)
res=Result(list_of_rewards,costs,episodes,running_time)
list_of_results.append(res)


# Best-checkpoint: evaluate the BEST policy seen in training
if getattr(RL_DQN_NC,'best_state_dict',None) is not None:
    RL_DQN_NC.q_network_policy.load_state_dict(RL_DQN_NC.best_state_dict)
    print('Loaded best checkpoint into RL_DQN_NC for evaluation.')

plot_dqnnc_test = display(display_id=True)
running_time,episodes,costs,list_of_rewards=train(env,RL_DQN_NC,max_iter=testing_max_iter,eps_decay=eps_decay,threshold=converge_threshold,path=path,train_enable=False,save_load=False,plot_handle=plot_dqnnc_test)
res=Result(list_of_rewards,costs,episodes,running_time)
pre_trained_results.append(res)

### DQN without caching + PBRS (nc_pbrs_v1)

In [ ]:

RL_DQN_NC_PBRS=DQNAgent_NC_PBRS(env,state_features=7,learning_rate=1e-4,gamma=gamma,eps=eps_value,max_iter=max_iter,replay_type="PER",replay_buffer_size=10_000,taf=0.005)

plot_dqnncpbrs_train = display(display_id=True)
running_time,episodes,costs,list_of_rewards=train(env,RL_DQN_NC_PBRS,max_iter=max_iter,eps_decay=eps_decay,threshold=converge_threshold,path=path,train_enable=True,save_load=False,plot_handle=plot_dqnncpbrs_train)
res_nc_pbrs_train=Result(list_of_rewards,costs,episodes,running_time)
list_of_results.append(res_nc_pbrs_train)

# Best-checkpoint: evaluate the BEST policy seen in training
if getattr(RL_DQN_NC_PBRS,'best_state_dict',None) is not None:
    RL_DQN_NC_PBRS.q_network_policy.load_state_dict(RL_DQN_NC_PBRS.best_state_dict)
    print('Loaded best checkpoint into RL_DQN_NC_PBRS for evaluation.')

plot_dqnncpbrs_test = display(display_id=True)
running_time,episodes,costs,list_of_rewards=train(env,RL_DQN_NC_PBRS,max_iter=testing_max_iter,eps_decay=eps_decay,threshold=converge_threshold,path=path,train_enable=False,save_load=False,plot_handle=plot_dqnncpbrs_test)
res_nc_pbrs_test=Result(list_of_rewards,costs,episodes,running_time)
pre_trained_results.append(res_nc_pbrs_test)

### DQN with caching

In [ ]:

plot_dqnwc_train = display(display_id=True)
RL_DQN_WC=DQNAgent_WC(env,state_features=7,learning_rate=1e-4,gamma=gamma,eps=eps_value,max_iter=max_iter,replay_type="PER",replay_buffer_size=10_000,taf=0.005)


running_time,episodes,costs,list_of_rewards=train(env,RL_DQN_WC,max_iter=max_iter,eps_decay=eps_decay,threshold=converge_threshold,path=path,train_enable=True,save_load=False,plot_handle=plot_dqnwc_train)
res=Result(list_of_rewards,costs,episodes,running_time)
list_of_results.append(res)

# Best-checkpoint: evaluate the BEST policy seen in training
if getattr(RL_DQN_WC,'best_state_dict',None) is not None:
    RL_DQN_WC.q_network_policy.load_state_dict(RL_DQN_WC.best_state_dict)
    print('Loaded best checkpoint into RL_DQN_WC for evaluation.')

plot_dqnwc_test = display(display_id=True)
running_time,episodes,costs,list_of_rewards=train(env,RL_DQN_WC,max_iter=testing_max_iter,eps_decay=eps_decay,threshold=converge_threshold,path=path,train_enable=False,save_load=False,plot_handle=plot_dqnwc_test)
res=Result(list_of_rewards,costs,episodes,running_time)
pre_trained_results.append(res)


## Results

In [ ]:
episode_slices=500


label=["RL Recommendation, popularity  based Caching","RL joint Recommendation  and Caching"]

graphic_compare_results_reward(list_of_results,eps_decay,"Cache Hit Rate",label=label,episode_slices=episode_slices,file_name="rewards.png")

graphic_compare_results_cost(list_of_results,eps_decay,"Loss",label=label,episode_slices=episode_slices,file_name="loss.png")

pre_trained_label=["Baseline","Greedy","RL Recommendation, popularity  based Caching","RL joint Recommendation  and Caching"]
graphic_compare_results_reward(pre_trained_results,eps_decay,"Cache Hit Rate",label=pre_trained_label,episode_slices=500,file_name="pre_trained_reward.png")


for i in range(len(list_of_results)):
    print(np.mean(list_of_results[i].reward),np.mean(list_of_results[i].cost),list_of_results[i].time//60)


for i in range(len(pre_trained_results)):
    print(np.mean(pre_trained_results[i].reward),np.mean(pre_trained_results[i].cost))



### Bars Plot

In [ ]:

label=["baseline","greedy","RL \n Recom/tion \n popularity \n based Caching","RL joint \n Recom/tion \n and Caching"]

agents_names=label
values=[]

for i in pre_trained_results:
  values.append(100*np.mean(i.reward))
colors = ['blue','orange', 'green','red' ]
# Create the bar graph
plt.bar(agents_names,values,color=colors)

for i, (name, value) in enumerate(zip(agents_names, values)):
  plt.annotate(f"{int(value)}", xy=(i, value), ha="center", va="bottom")  # Adjust ha/va if needed

plt.ylim((0,100))
plt.xlabel('Agent Type')
plt.ylabel('Cache hit rate %')
plt.title('Comparison of Agent Performance')
plt.savefig('pre_trained_bars.png')
plt.show()

In [ ]:
#  Auto-log the WHOLE experiment  
#  combined plots for all agents (hit-rate, loss, bars) + per-agent params & NN architecture
# from experiment_logger import log_experiment

# log_experiment(
#     pre_trained_results,                        # one eval Result per agent -> hit-rate curve + bars
#     labels=pre_trained_label,                   # legend name per agent (the Results-section legend)
#     agents=[None, None, RL_DQN_NC, RL_DQN_WC],  # parallel to pre_trained_results; None = non-RL heuristic
#     env=env,
#     train_results=list_of_results,              # RL training Results (NC, WC) -> loss curve + collapse
#     seed=7,                                     # matches set_seed(...)
#     train_episodes=max_iter, eval_episodes=testing_max_iter,
#     title="WC vs NC vs non-RL",
#     note="Real Data movielens continous caching no random initialization",                                    # <-- describe what changed this run
# )

In [ ]:
# policy_wo_cahing=[]
# for i in range(len(env_tab_1.all_states)):
#     state=find_state_by_id_faster(env_tab_1.cache_states,i)
#     next_state=np.argsort(RL_1.q_table[i])[-1],state[1]
#     print(state,"--->",next_state)
#     policy_wo_cahing.append((state,"--->",next_state))

In [ ]:
# policy_caching=[]
# print(u)
# for i in range(len(env_tab_2.all_states)):
#     state=find_state_by_id_faster(env_tab_2.cache_states,i)
#     next_state=action_format(cache_size,state,np.argsort(RL_2.q_table[i])[-1])
#     print(state,"--->",next_state,np.max(RL_2.q_table[i]))
#     policy_caching.append((state,"----->",next_state))

# PI vs Q-learning

In [ ]:

# policy_caching=[]
# for i in range(len(env_tab_2.all_states)):
#     state=find_state_by_id_faster(env_tab_2.cache_states,i)
#     next_state=action_format_pi(cache_size,state,np.argmax(RL_2.q_table[i]))
#     next_state_id=env_tab_2.all_states[next_state]
#     policy_caching.append((state,"----->",next_state,"-----",PI_agent.value[next_state_id]))



# counter=0
# number=0
# for i in range(len(env_tab_2.all_states)):
#     if(policy_caching[i][2]!=action_format_pi(PI_agent.cache_size,find_state_by_id_faster(PI_agent.cache_states,i),PI_agent.policy[i])):
#         next_state=action_format_pi(PI_agent.cache_size,find_state_by_id_faster(PI_agent.cache_states,i),PI_agent.policy[i])
#         id=env_tab_2.all_states[next_state]
#         number+=1
#         print(policy_caching[i],"-------------------",next_state,PI_agent.value[id])
#         if(policy_caching[i][4]!=PI_agent.value[id]):
#             counter+=1

# print(env_tab_2.number_of_states-number,"/",env_tab_2.number_of_states)
# print(env_tab_2.number_of_states-counter,"/",env_tab_2.number_of_states)